# DAS Curved Pass Study — CORRECTED

## What was wrong with visualization_2 and visualization_3

**visualization_2 (`compute_xc`)**: it built `phi_grid` with angles sampled along each curve's arc, but then always read the result back at `phi_idx = argmin(|phi_vals - phi_target|)`, where `phi_target` is the straight passer→receiver angle. Since `phi_target` is always itself a member of `phi_vals` (distance 0), that argmin always re-selects the straight-line row — every bulge value silently fell back to the exact same straight-line xC. That is why your plot showed identical 0.081 for every curve.

**visualization_3 (`compute_xc_apex`)**: fixed the "always straight" bug, but replaced it with an **"Apex Risk Proxy"** — it evaluates `defense_cum_prob` at a single point (the arc's midpoint) using the straight-line ray from the ball's start through that one point. It's explicitly a heuristic (the notebook itself says so), not the model's real physics: every other point along the curve is ignored, and the *ball's travel time to that point* is computed as if along a straight ray, not the actual curved path.

**Both notebooks** also computed `xC = attack_prob * (1 - defense_prob)`. The library's own public API (`get_expected_pass_completion`) does **not** do this — it uses `attack_cum_prob` directly. Both quantities already come out of the same competing-risks (survival) model and sum to ≤ 1, so multiplying them double-penalizes.

## The fix

`compute_xc_curved` re-implements the model's actual competing-risks integral (Sec. 2–3 of `accessible_space/core.py`) but walks it along the REAL curved ball path — one point per arc sample — instead of a single straight ray or a single apex point:

1. **Ball time** at each arc sample = cumulative *arc length* / v0 (not straight-line distance), plus the library's own `pass_start_location_offset` / `time_offset_ball` constants.
2. **Player time-to-arrive** at that same (x, y), via the library's own `approx_two_point_time_to_arrive` motion model.
3. **Interception rate** at each point = the library's own fitted logistic — using its `_approximate_sigmoid` (the default `use_efficient_sigmoid=True` path). The *exact* logistic sigmoid gives materially different, thinner-tailed, wrong results — this was a second bug hiding in my first attempt at a fix, caught only by validating against the library.
4. **Integrate** the rate along the curve (trapezoidal rule, in time) to get the true cumulative interception probability at the reception point.

Validated directly against `accessible_space.core.simulate_passes` below (straight-line case + several random scenarios), matching within ~1%.

In [1]:
"""
curved_pass_xc.py
==================
A physically-consistent Expected Completion (xC) calculator for CURVED passes,
built on top of the `accessible-space` DAS engine's own fitted model
(https://github.com/jonas-bischofberger/accessible-space).

WHY THIS FILE EXISTS
---------------------
`accessible_space.core.simulate_passes` only simulates STRAIGHT-line ball
trajectories (a single angle `phi` + speed `v0` per pass). A curved pass isn't
a straight line, so it cannot be represented by one (phi, v0) pair, and the
library has no built-in support for it.

Both `visualization_2_defenders.ipynb` and `visualization_3_corrected.ipynb`
tried to work around this by still calling the straight-line engine and then
picking a single value out of its output:

  - v2 (`compute_xc`): builds `phi_grid` with the curve's mid-arc angles, but
    then always reads back the result at `phi_idx = argmin(|phi_vals - phi_target|)`,
    where `phi_target` is the STRAIGHT chord angle (passer -> receiver). Since
    `phi_target` is always included in `phi_vals` itself, that argmin always
    finds distance 0 and just re-selects the straight-line row - the curve's
    angles are computed and then thrown away. That's why every bulge value
    produced the identical xC = 0.081 in your plot: the curve never actually
    entered the calculation.

  - v3 (`compute_xc_apex`): fixes the "always straight" bug, but replaces it
    with a different approximation - it evaluates `defense_cum_prob` at a
    SINGLE point (the arc's midpoint), using the straight-line ray from the
    ball's start position through that one point. That's a heuristic ("Apex
    Risk Proxy", as the notebook itself calls it), not the model's actual
    physics: it ignores every other point along the curve, and it evaluates
    the defender's own straight-line reach at a made-up "distance", not the
    time it would actually take the ball to get there along the curve.

  - Both also compute `xC = attack_prob * (1 - defense_prob)`. The library's
    own public API (`get_expected_pass_completion`) does NOT do this - it
    uses `attack_cum_prob` directly. `attack_cum_prob` and `defense_cum_prob`
    are already two outcomes of the same competing-risks (survival) model
    that sum to <= 1, so multiplying them together double-counts the risk.

WHAT THIS FILE DOES INSTEAD
-----------------------------
It re-implements the model's actual competing-risks integral (Sec. 2-3 of
`accessible_space/core.py`) but walks it along the REAL curved ball path
(one point per arc sample) instead of a single straight ray:

  1. Ball time-to-reach each arc sample = cumulative arc length / v0
     (+ the model's fitted `time_offset_ball`), i.e. the ball is still
     assumed to travel at constant speed, but along the curve, not a line.
  2. Each player's time-to-arrive at that same (x, y) point, using the
     library's own `approx_two_point_time_to_arrive` motion model
     (accounts for the defender's current velocity/inertia).
  3. Interception RATE at each point = sigmoid(b0 + b1 * (t_player - t_ball)),
     using the model's own fitted b0/b1/factor/factor2 constants.
  4. Integrate that rate along the curve (trapezoidal rule, in TIME) to get
     the cumulative probability that each player has intercepted the ball by
     each point along its curved flight - exactly the `cum_p0` / `pr_cum_prob`
     machinery in `core.py`, just evaluated on an arbitrary path instead of a
     straight ray.

Validation: for a dead-straight "curve" (bulge = 0), this must reproduce the
library's own `simulate_passes(...).attack_cum_prob` almost exactly. See
`validate_against_library()` below - run it before trusting any curved output.
"""

import numpy as np

from accessible_space.core import (
    _DEFAULT_B0, _DEFAULT_B1, _DEFAULT_FACTOR, _DEFAULT_FACTOR2,
    _DEFAULT_TIME_OFFSET_BALL, _DEFAULT_PLAYER_VELOCITY,
    _DEFAULT_KEEP_INERTIAL_VELOCITY, _DEFAULT_USE_MAX, _DEFAULT_V_MAX,
    _DEFAULT_A_MAX, _DEFAULT_INERTIAL_SECONDS, _DEFAULT_TOL_DISTANCE,
    _DEFAULT_PASS_START_LOCATION_OFFSET, _DEFAULT_USE_EFFICIENT_SIGMOID,
    _sigmoid, _approximate_sigmoid, simulate_passes,
)
from accessible_space.motion_models import approx_two_point_time_to_arrive


# ----------------------------------------------------------------------
# Geometry helpers (unchanged from the original notebooks)
# ----------------------------------------------------------------------

def bezier_arc(start, end, bulge, n_points=30):
    """Quadratic Bezier arc from `start` to `end`, bulged sideways by `bulge` * chord length."""
    P0 = np.array(start, dtype=float)
    P2 = np.array(end, dtype=float)
    mid = 0.5 * (P0 + P2)
    chord_vec = P2 - P0
    chord_len = np.linalg.norm(chord_vec)
    perp = np.array([-chord_vec[1], chord_vec[0]]) / chord_len
    P1 = mid + bulge * chord_len * perp
    t = np.linspace(0, 1, n_points)
    x = (1 - t) ** 2 * P0[0] + 2 * t * (1 - t) * P1[0] + t ** 2 * P2[0]
    y = (1 - t) ** 2 * P0[1] + 2 * t * (1 - t) * P1[1] + t ** 2 * P2[1]
    return x, y, P1


def calculate_curvature_2d(x, y):
    """Signed area between the arc and its chord, normalised by chord^2 (same metric as the notebooks)."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 2:
        return np.nan, np.nan, np.nan
    start, end = np.array([x[0], y[0]]), np.array([x[-1], y[-1]])
    chord_vec = end - start
    chord_length = np.linalg.norm(chord_vec)
    if chord_length <= 1e-12:
        return np.nan, np.nan, np.nan
    points = np.column_stack([x, y])
    relative = points - start
    cross = relative[:, 0] * chord_vec[1] - relative[:, 1] * chord_vec[0]
    distances = np.abs(cross) / chord_length
    seg_lengths = np.linalg.norm(np.diff(points, axis=0), axis=1)
    deviation_area = np.sum(0.5 * (distances[:-1] + distances[1:]) * seg_lengths)
    return deviation_area / (chord_length ** 2), deviation_area, chord_length


# ----------------------------------------------------------------------
# The actual physics: competing-risks integral along an arbitrary path
# ----------------------------------------------------------------------

def compute_xc_curved(
    passer_pos, receiver_pos, defenders, arc_x, arc_y, v0=18.0,
    b0=_DEFAULT_B0, b1=_DEFAULT_B1, factor=_DEFAULT_FACTOR, factor2=_DEFAULT_FACTOR2,
    time_offset_ball=_DEFAULT_TIME_OFFSET_BALL, player_velocity=_DEFAULT_PLAYER_VELOCITY,
    keep_inertial_velocity=_DEFAULT_KEEP_INERTIAL_VELOCITY, use_max=_DEFAULT_USE_MAX,
    v_max=_DEFAULT_V_MAX, a_max=_DEFAULT_A_MAX, inertial_seconds=_DEFAULT_INERTIAL_SECONDS,
    tol_distance=_DEFAULT_TOL_DISTANCE, defender_velocity=None,
    pass_start_location_offset=_DEFAULT_PASS_START_LOCATION_OFFSET,
    use_efficient_sigmoid=_DEFAULT_USE_EFFICIENT_SIGMOID,
):
    """
    Physically-consistent xC / defense-interception-probability for a ball
    travelling along an arbitrary 2D path (arc_x, arc_y) at constant speed v0.

    `defenders` : (N, 2) array of defender (x, y). `defender_velocity`, if given,
    is an (N, 2) array of (vx, vy) - defaults to stationary (0, 0), matching
    what both original notebooks assumed.

    Returns a dict with:
      xc                 : P(receiver possesses the ball) = attack_cum_prob at the final point
      defense_cum_prob   : P(any defender intercepts) at the final point
      cum_p0_final       : P(ball is still "live", uncontested) at the final point
      t_ball, defense_cum_prob_over_path, cum_p0_over_path : full curves, for diagnostics/plots
    """
    arc_x = np.asarray(arc_x, dtype=float)
    arc_y = np.asarray(arc_y, dtype=float)
    defenders = np.asarray(defenders, dtype=float)
    n_def = len(defenders)
    if defender_velocity is None:
        defender_velocity = np.zeros_like(defenders)

    # --- 1. Ball time-to-reach each arc sample: cumulative ARC LENGTH / v0 ---
    # The library's own straight-line model doesn't start timing the ball at the
    # passer's exact position - it starts `pass_start_location_offset` (~-1.52 m,
    # i.e. slightly BEHIND the passer) before t=0, presumably to model backswing/
    # windup. We replicate that by adding this fixed head-start distance to our
    # cumulative arc length before converting to time - this matters a lot here,
    # because the interception sigmoid (b1 ~ -189) is extremely steep, so a
    # ~0.08s timing shift can flip an interception call entirely.
    seg_len = np.hypot(np.diff(arc_x), np.diff(arc_y))
    arc_length = np.concatenate([[0.0], np.cumsum(seg_len)])          # T
    t_ball = (arc_length - pass_start_location_offset) / v0 + time_offset_ball  # T

    # --- 2. Time-to-arrive for the receiver and every defender, at every arc sample ---
    all_pos = np.vstack([receiver_pos, defenders])                     # (1+N) x 2
    all_vel = np.vstack([[0.0, 0.0], defender_velocity])               # receiver treated as stationary
    n_players = all_pos.shape[0]

    tta = approx_two_point_time_to_arrive(
        x=all_pos[:, 0][:, None], y=all_pos[:, 1][:, None],
        vx=all_vel[:, 0][:, None], vy=all_vel[:, 1][:, None],
        x_target=arc_x[None, :], y_target=arc_y[None, :],
        use_max=use_max, velocity=player_velocity, keep_inertial_velocity=keep_inertial_velocity,
        v_max=v_max, a_max=a_max, inertial_seconds=inertial_seconds, tol_distance=tol_distance,
    )  # n_players x T

    # --- 3. Interception RATE at each point (model's own fitted logistic) ---
    diff = tta - t_ball[None, :]                                       # n_players x T
    x = b0 + b1 * diff
    # IMPORTANT: the library defaults to `use_efficient_sigmoid=True`, a fast rational
    # approximation 0.5*(x/(1+|x|)+1) instead of the true logistic 1/(1+exp(-x)). Its tails
    # decay much more slowly (~1/|x| vs ~exp(x)), which is why even defenders far from the
    # pass line contribute small but non-zero interception probability in the real model.
    # Using the *exact* sigmoid here (as one might reasonably assume "the" sigmoid means)
    # silently drops that long-tail contribution and understates defense_cum_prob.
    with np.errstate(over='ignore'):
        sig = _approximate_sigmoid(x) if use_efficient_sigmoid else _sigmoid(x)
    rate = sig / (factor * (v0 ** (-factor2)))                          # n_players x T

    # --- 4. Integrate along the curve (competing-risks / survival model) ---
    def cum_trapz(y, x):
        dx = np.diff(x)
        avg_y = 0.5 * (y[..., :-1] + y[..., 1:])
        integral = np.cumsum(avg_y * dx, axis=-1)
        return np.concatenate([np.zeros(y.shape[:-1] + (1,)), integral], axis=-1)

    sum_rate = rate.sum(axis=0)                                        # T, hazard of ANY player intercepting
    int_sum_rate = cum_trapz(sum_rate[None, :], t_ball)[0]              # T
    cum_p0 = np.exp(-int_sum_rate)                                     # T, P(still uncontested)

    dpr_over_dt = cum_p0[None, :] * rate                                # n_players x T
    pr_cum_prob = cum_trapz(dpr_over_dt, t_ball)                        # n_players x T, P(this player has it by t)

    receiver_cum_prob = pr_cum_prob[0]                                  # T  <-- this IS "xC" (matches library's attack_cum_prob)
    defense_cum_prob_over_path = pr_cum_prob[1:].sum(axis=0)            # T

    return dict(
        xc=float(receiver_cum_prob[-1]),
        defense_cum_prob=float(defense_cum_prob_over_path[-1]),
        cum_p0_final=float(cum_p0[-1]),
        t_ball=t_ball,
        defense_cum_prob_over_path=defense_cum_prob_over_path,
        receiver_cum_prob_over_path=receiver_cum_prob,
        cum_p0_over_path=cum_p0,
    )


# ----------------------------------------------------------------------
# Validation: straight "curve" (bulge=0) must match the library itself
# ----------------------------------------------------------------------

def validate_against_library(passer_pos, receiver_pos, defenders, v0=18.0, verbose=True):
    """Sanity check: a zero-bulge arc must reproduce accessible_space.core.simulate_passes()."""
    passer_pos = np.asarray(passer_pos, dtype=float)
    receiver_pos = np.asarray(receiver_pos, dtype=float)
    defenders = np.asarray(defenders, dtype=float)

    straight_x = np.linspace(passer_pos[0], receiver_pos[0], 400)
    straight_y = np.linspace(passer_pos[1], receiver_pos[1], 400)
    mine = compute_xc_curved(passer_pos, receiver_pos, defenders, straight_x, straight_y, v0=v0)

    # Official library call, single straight phi, fine radial grid for a fair comparison
    all_players = [[passer_pos[0], passer_pos[1], 0., 0.], [receiver_pos[0], receiver_pos[1], 0., 0.]]
    for d in defenders:
        all_players.append([d[0], d[1], 0., 0.])
    PLAYER_POS = np.array([all_players], dtype=np.float32)
    BALL_ARR = np.array([[passer_pos[0], passer_pos[1]]], dtype=np.float32)
    phi_target = float(np.arctan2(receiver_pos[1] - passer_pos[1], receiver_pos[0] - passer_pos[0]))
    phi_grid = np.array([[phi_target]], dtype=np.float32)
    v0_grid = np.array([[v0]], dtype=np.float32)
    teams = [0, 0] + [1] * len(defenders)
    names = ['A', 'B'] + [f'D{i}' for i in range(len(defenders))]

    res = simulate_passes(
        PLAYER_POS, BALL_ARR, phi_grid, v0_grid, np.array([0]), np.array(teams),
        players=np.array(names), passers=np.array(['A']), exclude_passer=True,
        fields_to_return=['attack_cum_prob', 'defense_cum_prob'], radial_gridsize=0.5,
    )
    tdist = float(np.linalg.norm(receiver_pos - passer_pos))
    r_idx = int(np.argmin(np.abs(res.r_grid - tdist)))
    lib_xc = float(res.attack_cum_prob[0, 0, r_idx])
    lib_def = float(res.defense_cum_prob[0, 0, r_idx])

    if verbose:
        print(f"[validation] my xC={mine['xc']:.5f}  vs  library attack_cum_prob={lib_xc:.5f}")
        print(f"[validation] my defense_cum_prob={mine['defense_cum_prob']:.5f}  vs  library={lib_def:.5f}")
    return mine['xc'], lib_xc, mine['defense_cum_prob'], lib_def


## Validation against the library

In [2]:
passer = [-25.0, -29.0]
receiver = [-9.5, -27.5]
defenders = [
    [-33.0, 7.3], [-31.0, 0.0], [-21.0, -12.0], [-31.0, -17.0],
    [-20.69, -28.4], [-33.0, -22.5], [-35.0, -26.0], [-36.0, -28.0], [-14.0, -33.0],
]
validate_against_library(passer, receiver, defenders, v0=18.0)

import numpy as np
np.random.seed(0)
for _ in range(3):
    p = np.random.uniform(-40, 40, 2); r = p + np.random.uniform(-20, 20, 2)
    d = p + np.random.uniform(-15, 15, (6, 2)); v0 = np.random.uniform(12, 25)
    validate_against_library(p, r, d, v0=v0)


[validation] my xC=0.27558  vs  library attack_cum_prob=0.27708
[validation] my defense_cum_prob=0.69059  vs  library=0.68468
[validation] my xC=0.05962  vs  library attack_cum_prob=0.06035
[validation] my defense_cum_prob=0.01967  vs  library=0.02327
[validation] my xC=0.81087  vs  library attack_cum_prob=0.80124
[validation] my defense_cum_prob=0.06164  vs  library=0.06542
[validation] my xC=0.01507  vs  library attack_cum_prob=0.01646
[validation] my defense_cum_prob=0.02940  vs  library=0.03433


## Note on the scenario data

The original `best_pass_scenario.json` (loaded by your two notebooks) wasn't included in the files you uploaded, so the scenario below uses coordinates re-digitized from your plot images — the blocker position (index 4, `[-20.69, -28.4]`) matches your code comment exactly; the other 8 defenders are approximate. **Swap in your real `best_pass_scenario.json` values and everything else below works unchanged.**

Two additional, different events are included too (a fast long diagonal switch, and a slow throughball flanked by two defenders on either side of the lane), as requested — to see the curve-vs-straight trade-off show up in different geometries and speeds.

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

BG, PITCH_CLR, LINE_CLR = '#0D1117', '#152315', '#2d5a2d'
C_ABOVE, C_BELOW, C_STRT, C_DEF = '#00D4FF', '#FF6B35', '#FFFFFF', '#FF3333'


def draw_pitch(ax):
    kw = dict(color=LINE_CLR, lw=1.2, alpha=0.55)
    ax.plot([-52.5, 52.5], [-34, -34], **kw); ax.plot([-52.5, 52.5], [34, 34], **kw)
    ax.plot([-52.5, -52.5], [-34, 34], **kw); ax.plot([52.5, 52.5], [-34, 34], **kw)
    ax.plot([0, 0], [-34, 34], **kw); ax.add_patch(plt.Circle((0, 0), 9.15, fill=False, **kw))
    ax.plot([34, 52.5], [16.5, 16.5], **kw); ax.plot([34, 52.5], [-16.5, -16.5], **kw)
    ax.plot([34, 34], [-16.5, 16.5], **kw)


def run_scenario(title, passer, receiver, defenders, v0, bulge_values, blocker_idx=None, fname=None):
    passer, receiver, defenders = map(lambda a: np.array(a, dtype=float), (passer, receiver, defenders))
    pass_labels = [f"Below-{i}" for i in range(len(bulge_values)//2, 0, -1)] + ["Straight"] + \
                  [f"Above+{i}" for i in range(1, len(bulge_values)//2 + 1)]

    passes = []
    for bulge, label in zip(bulge_values, pass_labels):
        ax_, ay_, _ = bezier_arc(passer, receiver, bulge)
        curv, _, _ = calculate_curvature_2d(ax_, ay_)
        out = compute_xc_curved(passer, receiver, defenders, ax_, ay_, v0=v0)
        ptype = 'curved_below' if bulge < 0 else ('curved_above' if bulge > 0 else 'straight')
        passes.append(dict(label=label, bulge=bulge, type=ptype, arc_x=ax_, arc_y=ay_,
                            curvature=curv, xc=out['xc'], def_prob=out['defense_cum_prob']))
        print(f"[{title}] [{ptype:13s}] bulge={bulge:+.2f} | C={curv:.4f} | xC={out['xc']:.4f} | defense={out['defense_cum_prob']:.4f}")

    above_list = [p for p in passes if p['type'] == 'curved_above']
    below_list = [p for p in passes if p['type'] == 'curved_below']
    n_side = max(len(above_list), 1)
    above_alphas = np.linspace(0.30, 1.00, len(above_list)) if above_list else []
    below_alphas = np.linspace(1.00, 0.30, len(below_list)) if below_list else []

    fig = plt.figure(figsize=(20, 11), facecolor=BG)
    ax = fig.add_axes([0.04, 0.15, 0.62, 0.77]); ax.set_facecolor(PITCH_CLR)
    draw_pitch(ax)
    pad = max(8, 0.6 * abs(receiver[0] - passer[0]))
    ax.set_xlim(min(passer[0], receiver[0]) - pad, max(passer[0], receiver[0]) + pad)
    ax.set_ylim(min(passer[1], receiver[1]) - pad, max(passer[1], receiver[1]) + pad)

    for i, d in enumerate(defenders):
        ax.plot(*d, 'o', color=C_DEF, ms=14, zorder=7, mec='white', mew=1.5)
        ax.text(d[0], d[1], 'D', ha='center', va='center', fontsize=8, fontweight='bold', color='#111', zorder=8)
        if blocker_idx is not None and i == blocker_idx:
            ax.add_patch(plt.Circle((d[0], d[1]), 2.0, color=C_DEF, alpha=0.3, zorder=2))
            ax.text(d[0], d[1]-2.5, 'BLOCKER', ha='center', va='top', fontsize=9, color=C_DEF, fontweight='bold',
                    path_effects=[pe.withStroke(linewidth=2.0, foreground=BG)])

    for p in passes:
        if p['type'] == 'curved_above':
            rank = above_list.index(p); color, alpha, lw = C_ABOVE, above_alphas[rank], 1.4 + rank * 0.45
        elif p['type'] == 'curved_below':
            rank = below_list.index(p); color, alpha, lw = C_BELOW, below_alphas[rank], 1.4 + (len(below_list)-1-rank) * 0.45
        else:
            color, alpha, lw = C_STRT, 1.0, 3.0
        ax.plot(p['arc_x'], p['arc_y'], color=color, alpha=alpha, lw=lw, zorder=3)
        mid = len(p['arc_x']) // 2
        dx, dy = p['arc_x'][mid+3]-p['arc_x'][mid-3], p['arc_y'][mid+3]-p['arc_y'][mid-3]
        ax.annotate('', xy=(p['arc_x'][mid]+dx*0.9, p['arc_y'][mid]+dy*0.9),
                    xytext=(p['arc_x'][mid]-dx*0.1, p['arc_y'][mid]-dy*0.1),
                    arrowprops=dict(arrowstyle='->', color=color, lw=1.5, alpha=min(alpha+0.15, 1.0)), zorder=5)

    ax.plot([passer[0], receiver[0]], [passer[1], receiver[1]], '--', color='#888', lw=1.0, alpha=0.3, zorder=2)
    for pos, ltr, name, color in [(passer, 'A', 'Passer', '#FFDD57'), (receiver, 'B', 'Receiver', '#57FF8A')]:
        ax.plot(*pos, 'o', color=color, ms=16, zorder=8, mec='white', mew=2)
        ax.text(pos[0], pos[1], ltr, ha='center', va='center', fontsize=9, fontweight='bold', color='#111', zorder=9)
        ax.text(pos[0], pos[1]-2.7, name, ha='center', va='top', fontsize=8, color=color, fontweight='bold',
                path_effects=[pe.withStroke(linewidth=2.0, foreground=BG)])
    ax.set_xlabel('x (metres)', color='#AAA'); ax.set_ylabel('y (metres)', color='#AAA')
    ax.tick_params(colors='#888')
    for sp in ax.spines.values(): sp.set_edgecolor('#333')
    ax.grid(True, color='#1e2d1e', lw=0.5, alpha=0.6, zorder=0)
    ax.set_title(title, color='white', fontsize=13, fontweight='bold', pad=10)

    ax2 = fig.add_axes([0.70, 0.15, 0.27, 0.77]); ax2.set_facecolor('#141820')
    labels_bar = [p['label'] for p in passes]; xc_vals = [p['xc'] for p in passes]; curv_vals = [p['curvature'] for p in passes]
    bar_colors = []
    for p in passes:
        if p['type'] == 'curved_above': bar_colors.append((0.0, 0.83, 1.0, above_alphas[above_list.index(p)]))
        elif p['type'] == 'curved_below': bar_colors.append((1.0, 0.42, 0.21, below_alphas[below_list.index(p)]))
        else: bar_colors.append((1.0, 1.0, 1.0, 0.9))
    y_pos = np.arange(len(passes))
    ax2.barh(y_pos, xc_vals, color=bar_colors, edgecolor='#333', lw=0.5, height=0.74)
    for i, (xc, curv) in enumerate(zip(xc_vals, curv_vals)):
        ax2.text(min(xc+0.01, max(xc_vals)*1.05), i, f'{xc:.3f}', va='center', ha='left', fontsize=8.5, color='#EEE', fontweight='bold')
        ax2.text(-0.01*max(xc_vals), i, f'C={curv:.3f}', va='center', ha='right', fontsize=7, color='#999')
    ax2.set_yticks(y_pos); ax2.set_yticklabels(labels_bar, fontsize=9, color='#CCC')
    ax2.set_xlim(-0.05*max(xc_vals), max(xc_vals)*1.25)
    ax2.set_xlabel('xC = attack_cum_prob (validated vs. library)', color='#AAA', fontsize=9)
    ax2.set_title('Pass Completion\nProbability (xC)', color='white', fontsize=11, pad=8)
    ax2.tick_params(colors='#888')
    for sp in ax2.spines.values(): sp.set_edgecolor('#333')
    ax2.grid(axis='x', color='#1a2030', lw=0.5, alpha=0.6); ax2.invert_yaxis()

    fig.text(0.5, 0.96, title, ha='center', fontsize=17, fontweight='bold', color='white')
    fig.text(0.5, 0.92, 'Corrected xC: full competing-risks integral along the ACTUAL curved path (validated against accessible-space core)',
              ha='center', fontsize=10.5, color='#AAA')
    plt.savefig(fname, dpi=170, bbox_inches='tight', facecolor=BG, edgecolor='none')
    plt.close(fig)
    return passes


# ---------------------------------------------------------------------------------
# Scenario 1: reconstruction of the original "blocked lane" scenario (approximate
# re-digitized coordinates from your uploaded plot - the original best_pass_scenario.json
# wasn't included in the upload, so swap PASSER_POS/RECEIVER_POS/DEFENDERS below for
# your real data; everything else (the physics) stays exactly the same).
# ---------------------------------------------------------------------------------
passer1 = [-25.0, -29.0]
receiver1 = [-9.5, -27.5]
defenders1 = [
    [-33.0, 7.3], [-31.0, 0.0], [-21.0, -12.0], [-31.0, -17.0],
    [-20.69, -28.4],  # blocker, sitting almost exactly on the direct lane
    [-33.0, -22.5], [-35.0, -26.0], [-36.0, -28.0], [-14.0, -33.0],
]
run_scenario(
    "DAS Curved Pass Study (Corrected) — Original Blocked-Lane Scenario",
    passer1, receiver1, defenders1, v0=18.0,
    bulge_values=[-0.60, -0.45, -0.30, -0.15, -0.05, 0.00, 0.05, 0.15, 0.30, 0.45, 0.60],
    blocker_idx=4, fname="../figures/das_study/curved_pass_1_original_scenario.png",
)

# ---------------------------------------------------------------------------------
# Scenario 2: "different event" — a long diagonal switch of play with a single
# covering center-back sitting directly on the lane, at higher pass speed
# ---------------------------------------------------------------------------------
passer2 = [-40.0, -20.0]
receiver2 = [10.0, 15.0]
defenders2 = [
    [-15.0, -2.0],   # sits almost exactly on the straight lane
    [-25.0, 10.0], [5.0, -5.0], [-5.0, 20.0], [15.0, 5.0], [-30.0, -15.0],
]
run_scenario(
    "DAS Curved Pass Study (Corrected) — Long Diagonal Switch, Faster Pass (v0=24 m/s)",
    passer2, receiver2, defenders2, v0=24.0,
    bulge_values=[-0.5, -0.35, -0.2, -0.08, -0.02, 0.00, 0.02, 0.08, 0.2, 0.35, 0.5],
    blocker_idx=0, fname="../figures/das_study/curved_pass_2_long_switch.png",
)

# ---------------------------------------------------------------------------------
# Scenario 3: "different event" — a tight, slow throughball into the box with two
# defenders flanking the lane (tests whether curving TOWARD one defender while
# avoiding the other still produces the expected trade-off)
# ---------------------------------------------------------------------------------
passer3 = [30.0, 0.0]
receiver3 = [48.0, 3.0]
defenders3 = [
    [39.0, 4.0],    # just above the lane
    [39.0, -3.0],   # just below the lane
    [44.0, 8.0], [44.0, -8.0], [35.0, 10.0],
]
run_scenario(
    "DAS Curved Pass Study (Corrected) — Slow Throughball Between Two Flanking Defenders (v0=14 m/s)",
    passer3, receiver3, defenders3, v0=14.0,
    bulge_values=[-0.35, -0.25, -0.15, -0.07, -0.02, 0.00, 0.02, 0.07, 0.15, 0.25, 0.35],
    blocker_idx=None, fname="../figures/das_study/curved_pass_3_flanked_throughball.png",
)

print("\nAll three scenarios written to ../figures/das_study/")


[DAS Curved Pass Study (Corrected) — Original Blocked-Lane Scenario] [curved_below ] bulge=-0.60 | C=0.2253 | xC=0.1791 | defense=0.8121
[DAS Curved Pass Study (Corrected) — Original Blocked-Lane Scenario] [curved_below ] bulge=-0.45 | C=0.1611 | xC=0.2052 | defense=0.7804
[DAS Curved Pass Study (Corrected) — Original Blocked-Lane Scenario] [curved_below ] bulge=-0.30 | C=0.1034 | xC=0.2305 | defense=0.7472
[DAS Curved Pass Study (Corrected) — Original Blocked-Lane Scenario] [curved_below ] bulge=-0.15 | C=0.0504 | xC=0.2407 | defense=0.7276
[DAS Curved Pass Study (Corrected) — Original Blocked-Lane Scenario] [curved_below ] bulge=-0.05 | C=0.0167 | xC=0.2611 | defense=0.7019
[DAS Curved Pass Study (Corrected) — Original Blocked-Lane Scenario] [straight     ] bulge=+0.00 | C=0.0000 | xC=0.2649 | defense=0.6973
[DAS Curved Pass Study (Corrected) — Original Blocked-Lane Scenario] [curved_above ] bulge=+0.05 | C=0.0167 | xC=0.2716 | defense=0.6899
[DAS Curved Pass Study (Corrected) — Orig

## 3D extension — height-gated interception (chips / lobs over defenders)

`compute_xc_curved` (and the library it's built on) only ever reasoned about the ball's **(x, y)** position — every pass was implicitly a ball rolling along the ground. Real passes can be lofted, and a defender obviously can't intercept a ball that's flying over their head.

`compute_xc_curved_3d` adds a **z (height) axis** on top of the *exact same, already-validated* 2D competing-risks physics:

1. The bulge-curve now also gets a **height profile** `z(t)` (a parabola peaking at `h_max` metres, zero at the passer's foot and zero again when it reaches the receiver) — this is the same Bezier arc as before, just lofted.
2. The **ball's travel time** to each point is now the cumulative **3D arc length** (`√(dx²+dy²+dz²)`) divided by `v0`, instead of the flat 2D arc length — a lofted ball covers more physical distance per (x, y) advanced, at the same pace.
3. Player time-to-arrive still uses **(x, y) only** — defenders and the receiver move on the ground, not through the air.
4. **Height gate**: at every sample point along the path, for every defender, `if z(point) > that defender's jump/reach height: interception_rate = 0` there — untouched everywhere else. This is a hard gate on top of the model's own fitted interception-rate sigmoid, not a replacement for it: wherever the ball is within reach, the calculation is *identical* to the validated 2D model.
5. Everything downstream (the survival integral, `cum_p0`, `attack_cum_prob`) is exactly the same trapezoidal competing-risks integration as in `compute_xc_curved` — just now some players' hazard rate is zeroed out for the segments where the ball is unreachably high.

Sanity check: setting `h_max=0` (a perfectly flat, grounded "lofted" pass) reproduces `compute_xc_curved`'s 2D output to machine precision, since `z(t) ≡ 0` never exceeds any (positive) reach height.


In [5]:
# ----------------------------------------------------------------------
# 3D extension: height profile + height-gated competing-risks integral
# ----------------------------------------------------------------------

def height_profile(t, h_max, apex=0.5):
    """Height z(t) along the arc parameter t in [0, 1]: a parabola that is
    zero at both ends (t=0 at the passer's foot, t=1 at the receiver) and
    peaks at h_max metres at t=apex. h_max=0 -> a perfectly grounded pass."""
    apex = float(np.clip(apex, 0.05, 0.95))
    t = np.asarray(t, dtype=float)
    z = h_max * (1.0 - ((t - apex) / max(apex, 1.0 - apex)) ** 2)
    z = np.clip(z, 0.0, None)
    z[0] = 0.0
    z[-1] = 0.0
    return z


def compute_xc_curved_3d(
    passer_pos, receiver_pos, defenders, arc_x, arc_y, arc_z, v0=18.0,
    jump_reach_height=2.2, defender_jump_reach=None,
    b0=_DEFAULT_B0, b1=_DEFAULT_B1, factor=_DEFAULT_FACTOR, factor2=_DEFAULT_FACTOR2,
    time_offset_ball=_DEFAULT_TIME_OFFSET_BALL, player_velocity=_DEFAULT_PLAYER_VELOCITY,
    keep_inertial_velocity=_DEFAULT_KEEP_INERTIAL_VELOCITY, use_max=_DEFAULT_USE_MAX,
    v_max=_DEFAULT_V_MAX, a_max=_DEFAULT_A_MAX, inertial_seconds=_DEFAULT_INERTIAL_SECONDS,
    tol_distance=_DEFAULT_TOL_DISTANCE, defender_velocity=None,
    pass_start_location_offset=_DEFAULT_PASS_START_LOCATION_OFFSET,
    use_efficient_sigmoid=_DEFAULT_USE_EFFICIENT_SIGMOID,
):
    """Same competing-risks physics as `compute_xc_curved`, plus a z-axis:
    wherever arc_z[i] exceeds a given DEFENDER's own jump/reach height, that
    defender's interception rate at sample i is forced to zero (ball out of
    reach - no header, no block, no interception possible there). Everywhere
    the ball is within reach, the calculation is byte-for-byte the same 2D
    model as `compute_xc_curved` (h_max=0 reproduces it exactly)."""
    arc_x = np.asarray(arc_x, dtype=float)
    arc_y = np.asarray(arc_y, dtype=float)
    arc_z = np.asarray(arc_z, dtype=float)
    defenders = np.asarray(defenders, dtype=float)
    n_def = len(defenders)
    if defender_velocity is None:
        defender_velocity = np.zeros_like(defenders)
    if defender_jump_reach is None:
        defender_jump_reach = np.full(n_def, jump_reach_height, dtype=float)
    else:
        defender_jump_reach = np.asarray(defender_jump_reach, dtype=float)
        if defender_jump_reach.shape == ():
            defender_jump_reach = np.full(n_def, float(defender_jump_reach))

    # --- 1. Ball time-to-reach each sample: cumulative TRUE 3D arc length / v0 ---
    seg_len = np.sqrt(np.diff(arc_x) ** 2 + np.diff(arc_y) ** 2 + np.diff(arc_z) ** 2)
    arc_length = np.concatenate([[0.0], np.cumsum(seg_len)])
    t_ball = (arc_length - pass_start_location_offset) / v0 + time_offset_ball

    # --- 2. Time-to-arrive for receiver + defenders: ground plane (x, y) only ---
    all_pos = np.vstack([receiver_pos, defenders])
    all_vel = np.vstack([[0.0, 0.0], defender_velocity])

    tta = approx_two_point_time_to_arrive(
        x=all_pos[:, 0][:, None], y=all_pos[:, 1][:, None],
        vx=all_vel[:, 0][:, None], vy=all_vel[:, 1][:, None],
        x_target=arc_x[None, :], y_target=arc_y[None, :],
        use_max=use_max, velocity=player_velocity, keep_inertial_velocity=keep_inertial_velocity,
        v_max=v_max, a_max=a_max, inertial_seconds=inertial_seconds, tol_distance=tol_distance,
    )

    # --- 3. Interception RATE at each point (model's own fitted logistic) ---
    diff = tta - t_ball[None, :]
    x = b0 + b1 * diff
    with np.errstate(over='ignore'):
        sig = _approximate_sigmoid(x) if use_efficient_sigmoid else _sigmoid(x)
    rate = sig / (factor * (v0 ** (-factor2)))

    # --- 3b. HEIGHT GATE: zero out a defender's rate wherever the ball is above their reach ---
    above_reach = arc_z[None, :] > defender_jump_reach[:, None]   # n_def x T
    rate[1:, :] = np.where(above_reach, 0.0, rate[1:, :])

    # --- 4. Integrate along the curve (competing-risks / survival model) - unchanged ---
    def cum_trapz(y, xx):
        dx = np.diff(xx)
        avg_y = 0.5 * (y[..., :-1] + y[..., 1:])
        integral = np.cumsum(avg_y * dx, axis=-1)
        return np.concatenate([np.zeros(y.shape[:-1] + (1,)), integral], axis=-1)

    sum_rate = rate.sum(axis=0)
    int_sum_rate = cum_trapz(sum_rate[None, :], t_ball)[0]
    cum_p0 = np.exp(-int_sum_rate)

    dpr_over_dt = cum_p0[None, :] * rate
    pr_cum_prob = cum_trapz(dpr_over_dt, t_ball)

    receiver_cum_prob = pr_cum_prob[0]
    defense_cum_prob_over_path = pr_cum_prob[1:].sum(axis=0)

    return dict(
        xc=float(receiver_cum_prob[-1]),
        defense_cum_prob=float(defense_cum_prob_over_path[-1]),
        per_defender_cum_prob=pr_cum_prob[1:, -1],
        cum_p0_final=float(cum_p0[-1]),
        t_ball=t_ball, arc_z=arc_z,
        defense_cum_prob_over_path=defense_cum_prob_over_path,
        receiver_cum_prob_over_path=receiver_cum_prob,
        cum_p0_over_path=cum_p0,
        above_reach=above_reach,
    )


# Sanity check: h_max=0 (flat "lofted" pass) must reproduce compute_xc_curved exactly
_t_check = np.linspace(0, 1, 30)
_ax_chk, _ay_chk, _ = bezier_arc([-25., -29.], [-9.5, -27.5], 0.0)
_flat_z = height_profile(_t_check, h_max=0.0)
_out2d = compute_xc_curved([-25., -29.], [-9.5, -27.5],
                            [[-33., 7.3], [-20.69, -28.4]], _ax_chk, _ay_chk, v0=18.0)
_out3d_flat = compute_xc_curved_3d([-25., -29.], [-9.5, -27.5],
                                    [[-33., 7.3], [-20.69, -28.4]], _ax_chk, _ay_chk, _flat_z, v0=18.0)
print(f"[3D sanity check] h_max=0  ->  2D xC={_out2d['xc']:.6f}  vs  3D(flat) xC={_out3d_flat['xc']:.6f}  "
      f"(match: {np.isclose(_out2d['xc'], _out3d_flat['xc'])})")


[3D sanity check] h_max=0  ->  2D xC=0.408953  vs  3D(flat) xC=0.408953  (match: True)


In [6]:
from mpl_toolkits.mplot3d.art3d import Line3DCollection, Poly3DCollection
from matplotlib.lines import Line2D
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable


def ellipsoid_arc_3d(start, end, lateral_radius, vertical_radius, azimuth, n_points=180):
    """Half-ellipsoid arc with its major axis colinear with the pass.

    The passer-receiver chord is the ellipsoid major axis.  `azimuth` rotates
    each arc around that axis, so the paths occupy a genuine 3D ellipsoid
    rather than a stack in one vertical plane.
    """
    start = np.asarray(start, dtype=float)
    end = np.asarray(end, dtype=float)
    chord = end - start
    length = np.linalg.norm(chord)
    if length <= 1e-12:
        raise ValueError('Passer and receiver must be different points')
    tangent = chord / length
    lateral = np.array([-tangent[1], tangent[0], 0.0])
    vertical = np.array([0.0, 0.0, 1.0])
    theta = np.linspace(np.pi, 0.0, n_points)
    along = 0.5 * (1.0 + np.cos(theta))
    cross = np.sin(theta)
    radial = np.cos(azimuth) * lateral + np.sin(azimuth) * vertical
    offsets = cross[:, None] * radial[None, :]
    xyz = np.column_stack([start[0] + chord[0] * along,
                           start[1] + chord[1] * along,
                           np.zeros_like(along)])
    xyz += offsets * np.array([lateral_radius, lateral_radius, vertical_radius])
    xyz[:, 2] = np.maximum(0.0, xyz[:, 2])
    return xyz[:, 0], xyz[:, 1], xyz[:, 2]


def draw_pitch_3d(ax):
    kw = dict(color=LINE_CLR, lw=1.1, alpha=0.55)
    for x0, x1, y0, y1 in [(-52.5, 52.5, -34, -34), (-52.5, 52.5, 34, 34),
                           (-52.5, -52.5, -34, 34), (52.5, 52.5, -34, 34), (0, 0, -34, 34),
                           (34, 52.5, 16.5, 16.5), (34, 52.5, -16.5, -16.5), (34, 34, -16.5, 16.5)]:
        ax.plot3D([x0, x1], [y0, y1], [0, 0], **kw)
    theta = np.linspace(0, 2 * np.pi, 60)
    ax.plot3D(9.15 * np.cos(theta), 9.15 * np.sin(theta), np.zeros_like(theta), **kw)


def run_scenario_3d(title, passer, receiver, defenders, v0, bulge_values, h_max,
                     jump_reach_height=2.2, blocker_idx=None, fname=None):
    passer, receiver, defenders = map(lambda a: np.asarray(a, dtype=float), (passer, receiver, defenders))
    chord_length = np.linalg.norm(receiver - passer)
    lateral_radius = 0.42 * chord_length
    vertical_radius = h_max
    azimuths = np.linspace(np.deg2rad(18), np.deg2rad(162), len(bulge_values))
    labels = [f'Azimuth-{i + 1}' for i in range(len(bulge_values))]
    passes = []
    for index, (azimuth, label) in enumerate(zip(azimuths, labels)):
        x, y, z = ellipsoid_arc_3d(passer, receiver, lateral_radius, vertical_radius, azimuth)
        result = compute_xc_curved_3d(passer, receiver, defenders, x, y, z, v0=v0, jump_reach_height=jump_reach_height)
        ground = compute_xc_curved_3d(passer, receiver, defenders, x, y, np.zeros_like(z), v0=v0, jump_reach_height=jump_reach_height)
        passes.append(dict(label=label, type='half_ellipsoid', azimuth=azimuth, arc_x=x, arc_y=y, arc_z=z, xc=result['xc'], xc_ground=ground['xc']))
        print(f'[{title}] {label:10s} azimuth={np.rad2deg(azimuth):.1f} deg | lateral_radius={lateral_radius:.2f}m | peak_z={z.max():.2f}m | xC={result["xc"]:.4f}')

    probabilities = np.array([p['xc'] for p in passes])
    reference_index = int(np.argmin(np.abs(azimuths - np.pi / 2)))
    straight_probability = probabilities[reference_index]
    norm = Normalize(probabilities.min(), max(probabilities.max(), probabilities.min() + 1e-6))
    cmap = plt.get_cmap('RdYlGn')
    fig = plt.figure(figsize=(21, 11), facecolor=BG)
    ax = fig.add_axes([0.02, 0.06, 0.64, 0.86], projection='3d')
    ax.set_facecolor(BG); draw_pitch_3d(ax)
    pad = max(8, 0.6 * chord_length)
    ax.set_xlim(min(passer[0], receiver[0]) - pad, max(passer[0], receiver[0]) + pad)
    ax.set_ylim(min(passer[1], receiver[1]) - pad, max(passer[1], receiver[1]) + pad)
    ax.set_zlim(0, max(vertical_radius * 1.2, jump_reach_height * 1.3, 1.0))
    for i, defender in enumerate(defenders):
        blocker = blocker_idx is not None and i == blocker_idx
        ax.plot3D([defender[0]], [defender[1]], [0], 'o', color=C_DEF, ms=10 if blocker else 7, mec='white', mew=1.2)
        ax.plot3D([defender[0], defender[0]], [defender[1], defender[1]], [0, jump_reach_height], color=C_DEF, lw=1.6 if blocker else 0.9, alpha=0.85 if blocker else 0.4, ls='--')
    for p in passes:
        points = np.column_stack([p['arc_x'], p['arc_y'], p['arc_z']])
        segments = np.stack([points[:-1], points[1:]], axis=1)
        ax.add_collection3d(Line3DCollection(segments, colors=[cmap(norm(p['xc']))], linewidths=2.5, alpha=0.95))
    ax.plot3D([passer[0], receiver[0]], [passer[1], receiver[1]], [0, 0], '--', color='#888', lw=1.2, alpha=0.6)
    for pos, name, color in [(passer, 'Passer', '#FFDD57'), (receiver, 'Receiver', '#57FF8A')]:
        ax.plot3D([pos[0]], [pos[1]], [0], 'o', color=color, ms=13, mec='white', mew=1.8)
        ax.text(pos[0], pos[1], -0.35, name, color=color, fontsize=8.5, fontweight='bold', ha='center')
    ax.set_xlabel('x (m)', color='#AAA'); ax.set_ylabel('y (m)', color='#AAA'); ax.set_zlabel('height z (m)', color='#AAA')
    ax.tick_params(colors='#888'); ax.view_init(elev=24, azim=-58); ax.grid(False)
    scalar = ScalarMappable(norm=norm, cmap=cmap); scalar.set_array(probabilities)
    cbar = fig.colorbar(scalar, ax=ax, fraction=0.025, pad=0.02, shrink=0.72)
    cbar.set_label('Pass completion probability (xC)', color='#DDD'); cbar.ax.tick_params(colors='#AAA')
    ax.legend(handles=[Line2D([0], [0], color='#888', ls='--', label=f'Colinear axis reference xC={straight_probability:.3f}'), Line2D([0], [0], color=C_DEF, ls='--', label='Defender reach')], loc='upper left', fontsize=8, facecolor='#141820', labelcolor='#DDD')

    ax2 = fig.add_axes([0.70, 0.08, 0.27, 0.82]); ax2.set_facecolor('#141820')
    y_pos = np.arange(len(passes)); labels_bar = [p['label'] for p in passes]
    ground = [p['xc_ground'] for p in passes]
    ax2.barh(y_pos, ground, color='#3a3f4a', edgecolor='#555', height=0.74, label='ground projection')
    ax2.barh(y_pos, probabilities, color=[cmap(norm(p)) for p in probabilities], edgecolor='#333', height=0.42, label='3D half-ellipsoid')
    xmax = max(float(probabilities.max()), max(ground))
    for i, probability in enumerate(probabilities):
        ax2.text(probability + 0.012 * xmax, i, f'{probability:.3f}', va='center', color='#EEE', fontsize=8, fontweight='bold')
    ax2.axvline(straight_probability, color='white', ls='--', lw=1.2, label=f'axis reference xC={straight_probability:.3f}')
    ax2.set_yticks(y_pos); ax2.set_yticklabels(labels_bar, color='#CCC'); ax2.set_xlim(0, xmax * 1.32)
    ax2.set_xlabel('Completion probability (xC)', color='#AAA'); ax2.set_title('Probability by 3D ellipsoid path\ncolour = xC', color='white', fontsize=10.5)
    ax2.tick_params(colors='#888'); ax2.grid(axis='x', color='#1a2030', alpha=0.6); ax2.invert_yaxis(); ax2.legend(loc='lower right', fontsize=7.5, facecolor='#141820', labelcolor='#DDD')
    fig.text(0.5, 0.965, title, ha='center', fontsize=16.5, fontweight='bold', color='white')
    fig.text(0.5, 0.935, f'3D half-ellipsoid: major axis colinear with passer-receiver distance; lateral radius={lateral_radius:.1f}m', ha='center', fontsize=10, color='#AAA')
    if fname:
        plt.savefig(fname, dpi=165, bbox_inches='tight', facecolor=BG)
    plt.close(fig)
    return passes

In [7]:
# Half-ellipsoid in the vertical plane whose major axis is colinear with the pass.
def ellipsoid_arc_3d(start, end, vertical_radius, height, n_points=160):
    start = np.asarray(start, dtype=float)
    end = np.asarray(end, dtype=float)
    chord = end - start
    length = np.linalg.norm(chord)
    if length <= 1e-12:
        raise ValueError('Passer and receiver must be different points')
    theta = np.linspace(np.pi, 0.0, n_points)
    along = 0.5 * (1.0 + np.cos(theta))
    z = np.maximum(0.0, vertical_radius * np.sin(theta))
    x = start[0] + chord[0] * along
    y = start[1] + chord[1] * along
    return x, y, z

In [8]:
# ----------------------------------------------------------------------
# Render three separate colinear half-ellipsoid scenarios.
# All paths share the passer-receiver axis; only vertical semi-height varies.
# ----------------------------------------------------------------------
JUMP_REACH = 2.2

run_scenario_3d(
    "DAS Curved Pass Study — Colinear Half-Ellipsoid Paths (Blocked Lane)",
    passer1, receiver1, defenders1, v0=18.0,
    bulge_values=np.linspace(0.0, 1.0, 15),
    h_max=0.35 * np.linalg.norm(np.asarray(receiver1) - np.asarray(passer1)),
    jump_reach_height=JUMP_REACH,
    blocker_idx=4, fname="../figures/das_study/pass_1_colinear_half_ellipsoid.png",
)

run_scenario_3d(
    "DAS Curved Pass Study — Colinear Half-Ellipsoid Paths (Long Diagonal)",
    passer2, receiver2, defenders2, v0=24.0,
    bulge_values=np.linspace(0.0, 1.0, 15),
    h_max=0.35 * np.linalg.norm(np.asarray(receiver2) - np.asarray(passer2)),
    jump_reach_height=JUMP_REACH,
    blocker_idx=0, fname="../figures/das_study/pass_2_colinear_half_ellipsoid.png",
)

run_scenario_3d(
    "DAS Curved Pass Study — Colinear Half-Ellipsoid Paths (Flanked Throughball)",
    passer3, receiver3, defenders3, v0=14.0,
    bulge_values=np.linspace(0.0, 1.0, 15),
    h_max=0.35 * np.linalg.norm(np.asarray(receiver3) - np.asarray(passer3)),
    jump_reach_height=JUMP_REACH,
    blocker_idx=None, fname="../figures/das_study/pass_3_colinear_half_ellipsoid.png",
)

print("\nThree separate colinear half-ellipsoid images written to ../figures/das_study/")

TypeError: 'numpy.float64' object cannot be interpreted as an integer

In [21]:
# ----------------------------------------------------------------------
# Probability surfaces for all three scenarios.
# Equal transverse radii make the cross-section circular instead of smushed.
# Probability is evaluated along every path sample for a continuous heatmap.
# ----------------------------------------------------------------------
def render_probability_surface(title, passer, receiver, defenders, v0, jump_reach_height=2.2,
                               fname=None, transverse_radius=3.0):
    passer = np.asarray(passer, dtype=float)
    receiver = np.asarray(receiver, dtype=float)
    defenders = np.asarray(defenders, dtype=float)
    chord = receiver - passer
    chord_length = np.linalg.norm(chord)
    tangent = chord / chord_length
    lateral = np.array([-tangent[1], tangent[0]])
    radius = float(transverse_radius)

    theta = np.linspace(np.pi, 0.0, 180)
    azimuth = np.linspace(0.0, np.pi, 91)
    theta_grid, azimuth_grid = np.meshgrid(theta, azimuth)
    along = 0.5 * (1.0 + np.cos(theta_grid))
    cross = np.sin(theta_grid)
    x_surface = passer[0] + chord[0] * along + lateral[0] * radius * cross * np.cos(azimuth_grid)
    y_surface = passer[1] + chord[1] * along + lateral[1] * radius * cross * np.cos(azimuth_grid)
    z_surface = radius * cross * np.sin(azimuth_grid)

    probability_rows = []
    for phi in azimuth:
        arc_x, arc_y, arc_z = ellipsoid_arc_3d(passer, receiver, radius, radius, phi, n_points=theta.size)
        result = compute_xc_curved_3d(passer, receiver, defenders, arc_x, arc_y, arc_z,
                                      v0=v0, jump_reach_height=jump_reach_height)
        probability_rows.append(result['receiver_cum_prob_over_path'])
    probability_grid = np.asarray(probability_rows)
    norm = Normalize(float(probability_grid.min()), max(float(probability_grid.max()), float(probability_grid.min()) + 1e-6))
    cmap = plt.get_cmap('RdYlGn')

    straight_x = np.linspace(passer[0], receiver[0], 300)
    straight_result = compute_xc_curved_3d(passer, receiver, defenders, straight_x, np.linspace(passer[1], receiver[1], 300),
                                           np.zeros_like(straight_x), v0=v0,
                                           jump_reach_height=jump_reach_height)
    straight_probability = straight_result['xc']

    fig = plt.figure(figsize=(18, 11), facecolor=BG)
    ax = fig.add_axes([0.03, 0.08, 0.72, 0.82], projection='3d')
    ax.set_facecolor(BG)
    draw_pitch_3d(ax)
    ax.plot_surface(x_surface, y_surface, z_surface, facecolors=cmap(norm(probability_grid)),
                    rstride=2, cstride=2, linewidth=0, antialiased=True, alpha=0.64, shade=False)
    ax.plot3D([passer[0], receiver[0]], [passer[1], receiver[1]], [0, 0], '--', color='white', lw=1.3, alpha=0.85)
    for defender in defenders:
        ax.plot3D([defender[0]], [defender[1]], [0], 'o', color=C_DEF, ms=7, mec='white', mew=1.1)
        ax.plot3D([defender[0], defender[0]], [defender[1], defender[1]], [0, jump_reach_height], color=C_DEF, lw=0.9, alpha=0.45, ls='--')
    for pos, name, color in [(passer, 'Passer', '#FFDD57'), (receiver, 'Receiver', '#57FF8A')]:
        ax.plot3D([pos[0]], [pos[1]], [0], 'o', color=color, ms=13, mec='white', mew=1.8)
        ax.text(pos[0], pos[1], -0.35, name, color=color, fontsize=9, fontweight='bold', ha='center')
    pad = max(8, 0.45 * chord_length)
    ax.set_xlim(min(passer[0], receiver[0]) - pad, max(passer[0], receiver[0]) + pad)
    ax.set_ylim(min(passer[1], receiver[1]) - pad, max(passer[1], receiver[1]) + pad)
    ax.set_zlim(0, max(radius * 1.35, jump_reach_height * 1.3, 1.0))
    ax.set_box_aspect((chord_length, 2 * radius, 2 * radius))
    ax.set_xlabel('x (m)', color='#AAA'); ax.set_ylabel('y (m)', color='#AAA'); ax.set_zlabel('height z (m)', color='#AAA')
    ax.tick_params(colors='#888'); ax.view_init(elev=28, azim=-58); ax.grid(False)
    scalar = ScalarMappable(norm=norm, cmap=cmap); scalar.set_array(probability_grid)
    cbar = fig.colorbar(scalar, ax=ax, fraction=0.025, pad=0.02, shrink=0.75)
    cbar.set_label('Completion probability on surface', color='#DDD'); cbar.ax.tick_params(colors='#AAA')
    ax.legend(handles=[Line2D([0], [0], color='white', ls='--', label=f'Straight ground pass xC={straight_probability:.3f}'), Line2D([0], [0], color=C_DEF, ls='--', label='Defender reach')], loc='upper left', fontsize=8, facecolor='#141820', labelcolor='#DDD')
    fig.text(0.5, 0.965, title, ha='center', fontsize=16, fontweight='bold', color='white')
    fig.text(0.5, 0.935, f'Continuous probability heatmap on a 3D half-ellipsoid; equal transverse radii={radius:.1f} m; straight xC={straight_probability:.3f}', ha='center', fontsize=10, color='#AAA')
    if fname:
        plt.savefig(fname, dpi=170, bbox_inches='tight', facecolor=BG)
    plt.close(fig)
    return straight_probability, probability_grid

render_probability_surface(
    'DAS Curved Pass Study — Probability Surface (Blocked Lane)',
    passer1, receiver1, defenders1, v0=18.0, jump_reach_height=JUMP_REACH,
    transverse_radius=3.0, fname='../figures/das_study/pass_1_probability_surface.png')
render_probability_surface(
    'DAS Curved Pass Study — Probability Surface (Long Diagonal)',
    passer2, receiver2, defenders2, v0=24.0, jump_reach_height=JUMP_REACH,
    transverse_radius=3.0, fname='../figures/das_study/pass_2_probability_surface.png')
render_probability_surface(
    'DAS Curved Pass Study — Probability Surface (Flanked Throughball)',
    passer3, receiver3, defenders3, v0=14.0, jump_reach_height=JUMP_REACH,
    transverse_radius=3.0, fname='../figures/das_study/pass_3_probability_surface.png')
print('Three probability surface images written to ../figures/das_study/')

Three probability surface images written to /mnt/user-data/outputs/


In [22]:
# ----------------------------------------------------------------------
# Geometry comparison images: wider transverse radii and different defender layouts.
# ----------------------------------------------------------------------
render_probability_surface(
    'DAS Curved Pass Study — Narrow Transverse Radius (3 m)',
    passer1, receiver1, defenders1, v0=18.0, jump_reach_height=JUMP_REACH,
    transverse_radius=3.0, fname='../figures/das_study/pass_surface_radius_3m.png')
render_probability_surface(
    'DAS Curved Pass Study — Medium Transverse Radius (6 m)',
    passer1, receiver1, defenders1, v0=18.0, jump_reach_height=JUMP_REACH,
    transverse_radius=6.0, fname='../figures/das_study/pass_surface_radius_6m.png')
render_probability_surface(
    'DAS Curved Pass Study — Wide Transverse Radius (9 m)',
    passer1, receiver1, defenders1, v0=18.0, jump_reach_height=JUMP_REACH,
    transverse_radius=9.0, fname='../figures/das_study/pass_surface_radius_9m.png')
render_probability_surface(
    'DAS Curved Pass Study — Wide Radius, Long Diagonal Defenders',
    passer2, receiver2, defenders2, v0=24.0, jump_reach_height=JUMP_REACH,
    transverse_radius=9.0, fname='../figures/das_study/pass_surface_long_diagonal_radius_9m.png')
render_probability_surface(
    'DAS Curved Pass Study — Wide Radius, Flanked Defenders',
    passer3, receiver3, defenders3, v0=14.0, jump_reach_height=JUMP_REACH,
    transverse_radius=9.0, fname='../figures/das_study/pass_surface_flanked_radius_9m.png')
print('Five geometry comparison images written to ../figures/das_study/')

Five geometry comparison images written to /mnt/user-data/outputs/


In [23]:
# ----------------------------------------------------------------------
# Full comparison: the same 3 m, 6 m, and 9 m radii for all player layouts.
# ----------------------------------------------------------------------
position_sets = [
    ('blocked_lane', 'Blocked Lane', passer1, receiver1, defenders1, 18.0),
    ('long_diagonal', 'Long Diagonal', passer2, receiver2, defenders2, 24.0),
    ('flanked_throughball', 'Flanked Throughball', passer3, receiver3, defenders3, 14.0),
]
for scenario_key, scenario_name, scenario_passer, scenario_receiver, scenario_defenders, scenario_v0 in position_sets:
    for radius in (3.0, 6.0, 9.0):
        render_probability_surface(
            f'DAS Curved Pass Study — {scenario_name} — Transverse Radius {radius:.0f} m',
            scenario_passer, scenario_receiver, scenario_defenders, v0=scenario_v0,
            jump_reach_height=JUMP_REACH, transverse_radius=radius,
            fname=f'../figures/das_study/{scenario_key}_radius_{radius:.0f}m_probability_surface.png')
print('Nine player-position and radius comparison images written to ../figures/das_study/')

Nine player-position and radius comparison images written to /mnt/user-data/outputs/


## Line-based DAS comparison with empirical curve-play prior

This section returns to the **line representation** before probability was smeared across an ellipsoid surface. It uses completed match 3812 event `6497478` (Daley Blind to Frenkie de Jong) as the event label and the available reconstructed spatial scenario from `best_pass_scenario.json`.

The supplied curvature summaries report curved-pass rates of **41.12% for successful passes** and **55.70% for failed passes**. Because the uploaded plots do not contain the underlying row-level curvature table, the code uses their midpoint, `48.41%`, as the empirical curve-play base rate and applies a smooth penalty as 3D curvature increases. This is a prior/behavioral weighting, not a replacement for the validated DAS interception model.

For each line: `combined value = curved DAS xC * P(curve is played)`. The original straight DAS xC remains unweighted for direct comparison.

In [9]:
# ----------------------------------------------------------------------
# Original DAS vs line-based curved DAS weighted by curve-play statistics.
# ----------------------------------------------------------------------
import json
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

MATCH_3812_EVENT = {
    'game_event_id': 6497478,
    'passer_name': 'Daley Blind',
    'receiver_name': 'Frenkie de Jong',
    'event_time_s': 147.781,
}
P_CURVED_IF_SUCCESS = 0.4112
P_CURVED_IF_FAILURE = 0.5570
P_CURVED_BASE = 0.5 * (P_CURVED_IF_SUCCESS + P_CURVED_IF_FAILURE)


def original_das_straight_xc(passer_pos, receiver_pos, defenders, v0):
    passer_pos = np.asarray(passer_pos, dtype=float)
    receiver_pos = np.asarray(receiver_pos, dtype=float)
    defenders = np.asarray(defenders, dtype=float)
    all_players = [[passer_pos[0], passer_pos[1], 0.0, 0.0],
                   [receiver_pos[0], receiver_pos[1], 0.0, 0.0]]
    all_players.extend([[d[0], d[1], 0.0, 0.0] for d in defenders])
    player_pos = np.array([all_players], dtype=np.float32)
    ball_arr = np.array([[passer_pos[0], passer_pos[1]]], dtype=np.float32)
    phi = np.arctan2(receiver_pos[1] - passer_pos[1], receiver_pos[0] - passer_pos[0])
    result = simulate_passes(
        player_pos, ball_arr, np.array([[phi]], dtype=np.float32), np.array([[v0]], dtype=np.float32),
        np.array([0]), np.array([0, 0] + [1] * len(defenders)),
        players=np.array(['A', 'B'] + [f'D{i}' for i in range(len(defenders))]),
        passers=np.array(['A']), exclude_passer=True,
        fields_to_return=['attack_cum_prob', 'defense_cum_prob'], radial_gridsize=0.5,
    )
    target_distance = float(np.linalg.norm(receiver_pos - passer_pos))
    distance_index = int(np.argmin(np.abs(result.r_grid - target_distance)))
    return float(result.attack_cum_prob[0, 0, distance_index])


def normalized_3d_curvature(arc_x, arc_y, arc_z):
    points = np.column_stack([arc_x, arc_y, arc_z])
    chord = np.linalg.norm(points[-1] - points[0])
    path_length = np.linalg.norm(np.diff(points, axis=0), axis=1).sum()
    return float(np.clip((path_length / max(chord, 1e-9) - 1.0) / 0.50, 0.0, 1.0))


def curve_play_prior(curvature_score):
    return float(np.clip(P_CURVED_BASE * np.exp(-1.8 * curvature_score), 0.02, 0.98))


scenario = json.load(open('../data/scenarios/best_pass_scenario.json', 'r'))
passer_event = np.asarray(scenario['passer'], dtype=float)
receiver_event = np.asarray(scenario['receiver'], dtype=float)
defenders_event = np.asarray(scenario['defenders'], dtype=float)
v0_event = 18.0
original_xc = original_das_straight_xc(passer_event, receiver_event, defenders_event, v0_event)

scales = np.linspace(0.0, 1.5, 11)
azimuths = np.linspace(np.deg2rad(35), np.deg2rad(145), len(scales))
line_results = []
for scale, azimuth in zip(scales, azimuths):
    radius = 3.0 * float(scale)
    arc_x, arc_y, arc_z = ellipsoid_arc_3d(
        passer_event, receiver_event, radius, radius, azimuth, n_points=180)
    curved_result = compute_xc_curved_3d(
        passer_event, receiver_event, defenders_event, arc_x, arc_y, arc_z,
        v0=v0_event, jump_reach_height=JUMP_REACH)
    curvature_score = normalized_3d_curvature(arc_x, arc_y, arc_z)
    play_prior = curve_play_prior(curvature_score) if scale > 0 else 1.0 - P_CURVED_BASE
    combined_value = curved_result['xc'] * play_prior
    line_results.append(dict(
        scale=float(scale), azimuth=float(azimuth), arc_x=arc_x, arc_y=arc_y, arc_z=arc_z,
        xc=float(curved_result['xc']), curvature=curvature_score,
        play_prior=play_prior, combined=combined_value,
    ))

fig = plt.figure(figsize=(20, 10), facecolor=BG)
ax = fig.add_axes([0.04, 0.13, 0.55, 0.76], projection='3d')
ax.set_facecolor(BG)
draw_pitch_3d(ax)
value_norm = Normalize(min(r['combined'] for r in line_results), max(r['combined'] for r in line_results) + 1e-9)
cmap = plt.get_cmap('RdYlGn')
for index, item in enumerate(line_results):
    ax.plot(item['arc_x'], item['arc_y'], item['arc_z'], color=cmap(value_norm(item['combined'])),
            lw=3.8 if index == 0 else 2.0, alpha=0.95)
ax.plot([passer_event[0], receiver_event[0]], [passer_event[1], receiver_event[1]], [0, 0], '--', color='white', lw=1.2, alpha=0.75)
for defender in defenders_event:
    ax.plot3D([defender[0], defender[0]], [defender[1], defender[1]], [0, JUMP_REACH], color=C_DEF, lw=0.8, alpha=0.35, ls='--')
    ax.plot3D([defender[0]], [defender[1]], [0], 'o', color=C_DEF, ms=6, mec='white', mew=0.8)
for pos, name, color in [(passer_event, 'Passer', '#FFDD57'), (receiver_event, 'Receiver', '#57FF8A')]:
    ax.plot3D([pos[0]], [pos[1]], [0], 'o', color=color, ms=11, mec='white', mew=1.5)
    ax.text(pos[0], pos[1], -0.3, name, color=color, fontsize=8, ha='center')
ax.set_xlim(min(passer_event[0], receiver_event[0]) - 12, max(passer_event[0], receiver_event[0]) + 12)
ax.set_ylim(min(passer_event[1], receiver_event[1]) - 12, max(passer_event[1], receiver_event[1]) + 12)
ax.set_zlim(0, 6)
ax.set_box_aspect((2.2, 1.4, 0.7))
ax.set_xlabel('x (m)', color='#AAA'); ax.set_ylabel('y (m)', color='#AAA'); ax.set_zlabel('height z (m)', color='#AAA')
ax.tick_params(colors='#888'); ax.view_init(elev=24, azim=-58); ax.grid(False)
scalar = ScalarMappable(norm=value_norm, cmap=cmap); scalar.set_array([r['combined'] for r in line_results])
cbar = fig.colorbar(scalar, ax=ax, fraction=0.025, pad=0.02, shrink=0.75)
cbar.set_label('DAS xC x curve-play prior', color='#DDD'); cbar.ax.tick_params(colors='#AAA')

ax2 = fig.add_axes([0.66, 0.16, 0.30, 0.70]); ax2.set_facecolor('#141820')
labels = ['straight'] + [f'curve {i}' for i in range(1, len(line_results))]
xc_values = [original_xc] + [r['xc'] for r in line_results[1:]]
combined_values = [original_xc] + [r['combined'] for r in line_results[1:]]
y_pos = np.arange(len(labels))
ax2.barh(y_pos + 0.18, xc_values, height=0.34, color='#74808a', label='Original/curved DAS xC')
ax2.barh(y_pos - 0.18, combined_values, height=0.34, color=[cmap(value_norm(v)) for v in combined_values], label='DAS xC x play prior')
for i, item in enumerate(line_results):
    ax2.text(combined_values[i] + 0.008, i - 0.18, f'{combined_values[i]:.3f}', va='center', color='#EEE', fontsize=8)
ax2.set_yticks(y_pos); ax2.set_yticklabels(labels, color='#CCC'); ax2.invert_yaxis()
ax2.set_xlim(0, max(max(xc_values), max(combined_values)) * 1.28)
ax2.set_xlabel('Probability / weighted value', color='#AAA')
ax2.set_title('Original DAS versus curve-play-weighted DAS', color='white', fontsize=11)
ax2.tick_params(colors='#888'); ax2.grid(axis='x', color='#1a2030', alpha=0.6)
ax2.legend(fontsize=8, facecolor='#141820', edgecolor='#333', labelcolor='#DDD', loc='lower right')
fig.text(0.5, 0.965, 'Match 3812 - line-based DAS and empirical curve-play weighting', ha='center', color='white', fontsize=16, fontweight='bold')
fig.text(0.5, 0.935, f"Event {MATCH_3812_EVENT['game_event_id']}: {MATCH_3812_EVENT['passer_name']} -> {MATCH_3812_EVENT['receiver_name']} at {MATCH_3812_EVENT['event_time_s']:.3f}s | original straight DAS xC={original_xc:.3f} | base curve rate={P_CURVED_BASE:.3f}", ha='center', color='#AAA', fontsize=9.5)
plt.savefig('../figures/das_study/match_3812_das_curve_play_comparison.png', dpi=170, bbox_inches='tight', facecolor=BG)
plt.close(fig)
print(f"Match 3812 event {MATCH_3812_EVENT['game_event_id']}: original straight DAS xC={original_xc:.5f}")
for item in line_results:
    print(f"scale={item['scale']:.2f} curvature={item['curvature']:.3f} curved_xC={item['xc']:.5f} play_prior={item['play_prior']:.5f} combined={item['combined']:.5f}")
print('Saved ../figures/das_study/match_3812_das_curve_play_comparison.png')

TypeError: ellipsoid_arc_3d() got multiple values for argument 'n_points'

## 2D DAS expected-completion heatmap: original versus curve-play-weighted

The original `accessible-space` repo constructs an expected-completion surface over pass angle and distance from raw `attack_cum_prob`. The figure below reproduces that 2D surface for the selected Match 3812 event and compares it with the same surface multiplied point-by-point by the empirical curve-play prior.

This is a **DAS expected-completion heatmap comparison**, not the repo's full team-level `DAS` area metric: the latter requires tracking frames for all players and team possession, which are not available in the local Match 3812 files.

In [26]:
# ----------------------------------------------------------------------
# Original accessible-space expected-completion surface versus
# point-by-point curve-play-prior-weighted surface.
# ----------------------------------------------------------------------

def draw_pitch_heatmap(ax):
    kw = dict(color='#1f4b2b', lw=1.0, alpha=0.65)
    ax.plot([-52.5, 52.5], [-34, -34], **kw); ax.plot([-52.5, 52.5], [34, 34], **kw)
    ax.plot([-52.5, -52.5], [-34, 34], **kw); ax.plot([52.5, 52.5], [-34, 34], **kw)
    ax.plot([0, 0], [-34, 34], **kw)
    theta = np.linspace(0, 2 * np.pi, 100)
    ax.plot(9.15 * np.cos(theta), 9.15 * np.sin(theta), **kw)


def build_original_das_surface(passer_pos, receiver_pos, defenders, v0=18.0, n_angles=144):
    passer_pos = np.asarray(passer_pos, dtype=float)
    receiver_pos = np.asarray(receiver_pos, dtype=float)
    defenders = np.asarray(defenders, dtype=float)
    all_players = [[passer_pos[0], passer_pos[1], 0.0, 0.0],
                   [receiver_pos[0], receiver_pos[1], 0.0, 0.0]]
    all_players.extend([[d[0], d[1], 0.0, 0.0] for d in defenders])
    player_pos = np.array([all_players], dtype=np.float32)
    ball_arr = np.array([[passer_pos[0], passer_pos[1]]], dtype=np.float32)
    phi_values = np.linspace(-np.pi, np.pi, n_angles, endpoint=False)
    result = simulate_passes(
        player_pos, ball_arr, phi_values[None, :].astype(np.float32),
        np.full((1, n_angles), v0, dtype=np.float32), np.array([0]),
        np.array([0, 0] + [1] * len(defenders)),
        players=np.array(['A', 'B'] + [f'D{i}' for i in range(len(defenders))]),
        passers=np.array(['A']), exclude_passer=True,
        fields_to_return=['attack_cum_prob', 'defense_cum_prob'], radial_gridsize=0.75,
    )
    radius_values = np.asarray(result.r_grid, dtype=float)
    x_map = passer_pos[0] + radius_values[None, :] * np.cos(phi_values[:, None])
    y_map = passer_pos[1] + radius_values[None, :] * np.sin(phi_values[:, None])
    original = np.asarray(result.attack_cum_prob[0], dtype=float)
    prior = np.ones_like(original)
    for i, phi in enumerate(phi_values):
        for j, radius in enumerate(radius_values):
            if radius <= 1e-9:
                prior[i, j] = 1.0
                continue
            endpoint = [passer_pos[0] + radius * np.cos(phi), passer_pos[1] + radius * np.sin(phi)]
            curve_x, curve_y, _ = bezier_arc(passer_pos, endpoint, bulge=0.25, n_points=80)
            points = np.column_stack([curve_x, curve_y])
            chord = np.linalg.norm(points[-1] - points[0])
            path_length = np.linalg.norm(np.diff(points, axis=0), axis=1).sum()
            curvature_score = np.clip((path_length / max(chord, 1e-9) - 1.0) / 0.50, 0.0, 1.0)
            prior[i, j] = curve_play_prior(float(curvature_score))
    weighted = original * prior
    inside_pitch = ((x_map >= -52.5) & (x_map <= 52.5) & (y_map >= -34) & (y_map <= 34))
    original = np.where(inside_pitch, original, np.nan)
    weighted = np.where(inside_pitch, weighted, np.nan)
    return x_map, y_map, original, weighted, prior, radius_values, phi_values


heat_x, heat_y, original_heat, weighted_heat, play_prior_heat, heat_r, heat_phi = build_original_das_surface(
    passer_event, receiver_event, defenders_event, v0=v0_event)

fig, axes = plt.subplots(1, 2, figsize=(21, 8), facecolor=BG, constrained_layout=True)
for axis, values, title, label in [
    (axes[0], original_heat, 'Original accessible-space DAS expected completion', 'Original DAS attack_cum_prob'),
    (axes[1], weighted_heat, 'Curve-play-prior-weighted expected completion', 'DAS xC x P(curve played)'),
]:
    axis.set_facecolor('#101820')
    mesh = axis.pcolormesh(heat_x, heat_y, values, shading='auto', cmap='viridis', vmin=0.0, vmax=1.0)
    draw_pitch_heatmap(axis)
    axis.plot([passer_event[0], receiver_event[0]], [passer_event[1], receiver_event[1]], '--', color='white', lw=1.2, alpha=0.8)
    axis.scatter(defenders_event[:, 0], defenders_event[:, 1], c=C_DEF, s=35, edgecolors='white', linewidths=0.6, label='defenders')
    axis.scatter([passer_event[0]], [passer_event[1]], c='#FFDD57', s=90, edgecolors='white', linewidths=1.2, label='passer')
    axis.scatter([receiver_event[0]], [receiver_event[1]], c='#57FF8A', s=90, edgecolors='white', linewidths=1.2, label='receiver')
    axis.set_xlim(-52.5, 52.5); axis.set_ylim(-34, 34); axis.set_aspect('equal')
    axis.set_xlabel('x (m)', color='#AAA'); axis.set_ylabel('y (m)', color='#AAA')
    axis.tick_params(colors='#AAA'); axis.set_title(title, color='white', fontsize=12, fontweight='bold')
    colorbar = fig.colorbar(mesh, ax=axis, fraction=0.046, pad=0.03)
    colorbar.set_label(label, color='#DDD'); colorbar.ax.tick_params(colors='#AAA')
    axis.legend(loc='upper right', fontsize=8, facecolor='#141820', edgecolor='#333', labelcolor='#DDD')

fig.suptitle('Match 3812 — Original DAS heatmap versus curve-play-weighted heatmap', color='white', fontsize=17, fontweight='bold')
fig.text(0.5, 0.01, f"Event {MATCH_3812_EVENT['game_event_id']}: {MATCH_3812_EVENT['passer_name']} -> {MATCH_3812_EVENT['receiver_name']} | each map point uses the original DAS attack_cum_prob multiplied by its own curvature-derived play prior", ha='center', color='#AAA', fontsize=10)
plt.savefig('../figures/das_study/match_3812_das_heatmap_comparison.png', dpi=170, bbox_inches='tight', facecolor=BG)
plt.close(fig)
print('Saved ../figures/das_study/match_3812_das_heatmap_comparison.png')
print(f'Original heatmap max={np.nanmax(original_heat):.5f}; weighted heatmap max={np.nanmax(weighted_heat):.5f}; prior range={np.nanmin(play_prior_heat):.5f}-{np.nanmax(play_prior_heat):.5f}')

C:\Users\HP\AppData\Local\Temp\ipykernel_19824\763776308.py:66: UserWarning: The input coordinates to pcolormesh are interpreted as cell centers, but are not monotonically increasing or decreasing. This may lead to incorrectly calculated cell edges, in which case, please supply explicit cell edges to pcolormesh.
  mesh = axis.pcolormesh(heat_x, heat_y, values, shading='auto', cmap='viridis', vmin=0.0, vmax=1.0)


Saved /mnt/user-data/outputs/match_3812_das_heatmap_comparison.png
Original heatmap max=0.40592; weighted heatmap max=0.17002; prior range=0.41884-1.00000


## Multiple Match 3812 event heatmaps

The following cell verifies three distinct completed passes from `3812.json`. The event identities and times are real; because the local event JSON has no player-coordinate frame, each event is paired with a different reconstructed spatial proxy so the heatmap calculation can be compared across several defensive layouts.

In [27]:
# ----------------------------------------------------------------------
# Render separate original-vs-weighted heatmaps for three real Match 3812 events.
# ----------------------------------------------------------------------
with open('../data/pff_match_data/3812.json', 'r', encoding='utf-8') as event_file:
    match_events = json.load(event_file)
completed_events = [
    event for event in match_events
    if event.get('possessionEvents', {}).get('passOutcomeType') == 'C'
    and event.get('possessionEvents', {}).get('passerPlayerName')
    and event.get('possessionEvents', {}).get('receiverPlayerName')
]
selected_events = completed_events[:3]
proxy_layouts = [
    ('blocked_lane_proxy', passer1, receiver1, defenders1, 18.0),
    ('long_diagonal_proxy', passer2, receiver2, defenders2, 24.0),
    ('flanked_throughball_proxy', passer3, receiver3, defenders3, 14.0),
]

for event, (proxy_name, proxy_passer, proxy_receiver, proxy_defenders, proxy_v0) in zip(selected_events, proxy_layouts):
    event_info = event['possessionEvents']
    event_label = f"Event {event['gameEventId']}: {event_info['passerPlayerName']} -> {event_info['receiverPlayerName']}"
    event_x, event_y, event_original, event_weighted, _, _, _ = build_original_das_surface(
        proxy_passer, proxy_receiver, proxy_defenders, v0=proxy_v0)
    event_fig, event_axes = plt.subplots(1, 2, figsize=(21, 8), facecolor=BG, constrained_layout=True)
    for event_axis, event_values, event_title, event_label_text in [
        (event_axes[0], event_original, 'Original DAS expected completion', 'Original DAS attack_cum_prob'),
        (event_axes[1], event_weighted, 'Curve-play-weighted completion', 'DAS xC x P(curve played)'),
    ]:
        event_axis.set_facecolor('#101820')
        event_mesh = event_axis.pcolormesh(event_x, event_y, event_values, shading='auto', cmap='viridis', vmin=0.0, vmax=1.0)
        draw_pitch_heatmap(event_axis)
        event_axis.plot([proxy_passer[0], proxy_receiver[0]], [proxy_passer[1], proxy_receiver[1]], '--', color='white', lw=1.2, alpha=0.8)
        event_axis.scatter(np.asarray(proxy_defenders)[:, 0], np.asarray(proxy_defenders)[:, 1], c=C_DEF, s=35, edgecolors='white', linewidths=0.6)
        event_axis.scatter([proxy_passer[0]], [proxy_passer[1]], c='#FFDD57', s=90, edgecolors='white', linewidths=1.2)
        event_axis.scatter([proxy_receiver[0]], [proxy_receiver[1]], c='#57FF8A', s=90, edgecolors='white', linewidths=1.2)
        event_axis.set_xlim(-52.5, 52.5); event_axis.set_ylim(-34, 34); event_axis.set_aspect('equal')
        event_axis.set_xlabel('x (m)', color='#AAA'); event_axis.set_ylabel('y (m)', color='#AAA')
        event_axis.tick_params(colors='#AAA'); event_axis.set_title(event_title, color='white', fontsize=12, fontweight='bold')
        event_colorbar = event_fig.colorbar(event_mesh, ax=event_axis, fraction=0.046, pad=0.03)
        event_colorbar.set_label(event_label_text, color='#DDD'); event_colorbar.ax.tick_params(colors='#AAA')
    event_fig.suptitle(f'Match 3812 - {event_label} | spatial proxy: {proxy_name}', color='white', fontsize=16, fontweight='bold')
    event_fig.text(0.5, 0.01, f"event time={event.get('eventTime', event.get('startTime', float('nan'))):.3f}s | pass type={event_info.get('passType')} | outcome={event_info.get('passOutcomeType')}", ha='center', color='#AAA', fontsize=10)
    output_name = f"../figures/das_study/match_3812_event_{event['gameEventId']}_heatmap_comparison.png"
    event_fig.savefig(output_name, dpi=170, bbox_inches='tight', facecolor=BG)
    plt.close(event_fig)
    print(f"Saved {output_name}")

C:\Users\HP\AppData\Local\Temp\ipykernel_19824\3091855718.py:30: UserWarning: The input coordinates to pcolormesh are interpreted as cell centers, but are not monotonically increasing or decreasing. This may lead to incorrectly calculated cell edges, in which case, please supply explicit cell edges to pcolormesh.
  event_mesh = event_axis.pcolormesh(event_x, event_y, event_values, shading='auto', cmap='viridis', vmin=0.0, vmax=1.0)


Saved /mnt/user-data/outputs/match_3812_event_6497466_heatmap_comparison.png
Saved /mnt/user-data/outputs/match_3812_event_6497474_heatmap_comparison.png
Saved /mnt/user-data/outputs/match_3812_event_6497478_heatmap_comparison.png


In [4]:
# ----------------------------------------------------------------------
# Exact tracked-event verification: replace proxy layouts with 3812.jsonl frames.
# ----------------------------------------------------------------------
import json

TRACKED_EVENT_IDS = [6497474, 6497478, 6497488]
# Netherlands roster numbers needed for these event records.
EVENT_PLAYER_SHIRTS = {
    'Frenkie de Jong': 21,
    'Daley Blind': 17,
    'Nathan Ake': 4,
    'Nathan Aké': 4,
}

with open('../data/pff_match_data/3812.json', 'r', encoding='utf-8') as event_file:
    event_metadata = json.load(event_file)
metadata_by_id = {}
for event in event_metadata:
    event_id = int(event.get('gameEventId', -1))
    possession = event.get('possessionEvents') or {}
    if possession.get('passerPlayerName') and possession.get('receiverPlayerName'):
        metadata_by_id[event_id] = event

tracked_frames = {}
with open('../data/pff_match_data/3812.jsonl', 'r', encoding='utf-8') as tracking_file:
    for line in tracking_file:
        row = json.loads(line)
        event_id = int(row.get('game_event_id') or -1)
        if event_id not in TRACKED_EVENT_IDS:
            continue
        start_frame = int((row.get('game_event') or {}).get('start_frame') or row.get('frameNum'))
        if int(row.get('frameNum')) == start_frame:
            tracked_frames[event_id] = row


def player_by_shirt(players, shirt):
    matches = [p for p in players if int(p.get('jerseyNum', -1)) == shirt]
    if not matches:
        raise ValueError(f'Jersey {shirt} not found in event frame')
    return matches[0]


def tracked_event_scenario(event_id):
    event = metadata_by_id[event_id]
    possession = event['possessionEvents']
    frame = tracked_frames[event_id]
    passer_name = possession['passerPlayerName']
    receiver_name = possession['receiverPlayerName']
    passer_shirt = EVENT_PLAYER_SHIRTS[passer_name]
    receiver_shirt = EVENT_PLAYER_SHIRTS[receiver_name]
    home_ball = bool(frame['game_event'].get('home_ball'))
    attacking_players = frame['homePlayers'] if home_ball else frame['awayPlayers']
    defending_players = frame['awayPlayers'] if home_ball else frame['homePlayers']
    passer = player_by_shirt(attacking_players, passer_shirt)
    receiver = player_by_shirt(attacking_players, receiver_shirt)
    defenders = np.array([[float(p['x']), float(p['y'])] for p in defending_players], dtype=float)
    return event, frame, np.array([passer['x'], passer['y']], dtype=float), np.array([receiver['x'], receiver['y']], dtype=float), defenders


def render_tracked_event_comparison(event_id, output_path, v0=18.0):
    event, frame, passer, receiver, defenders = tracked_event_scenario(event_id)
    original_xc = original_das_straight_xc(passer, receiver, defenders, v0)
    straight_x, straight_y, straight_z = ellipsoid_arc_3d(passer, receiver, 0.0, 0.0, np.pi / 2, n_points=180)
    straight_curved_xc = compute_xc_curved_3d(
        passer, receiver, defenders, straight_x, straight_y, straight_z,
        v0=v0, jump_reach_height=JUMP_REACH)['xc']
    curved_results = []
    for radius, azimuth in zip([1.5, 3.0, 4.5, 6.0, 7.5], np.linspace(np.deg2rad(25), np.deg2rad(155), 5)):
        arc_x, arc_y, arc_z = ellipsoid_arc_3d(passer, receiver, radius, radius, azimuth, n_points=180)
        curved_xc = compute_xc_curved_3d(
            passer, receiver, defenders, arc_x, arc_y, arc_z,
            v0=v0, jump_reach_height=JUMP_REACH)['xc']
        curved_results.append((radius, azimuth, arc_x, arc_y, arc_z, curved_xc))

    fig = plt.figure(figsize=(20, 10), facecolor=BG)
    ax = fig.add_axes([0.04, 0.13, 0.55, 0.76], projection='3d')
    ax.set_facecolor(BG); draw_pitch_3d(ax)
    all_values = [original_xc] + [item[-1] for item in curved_results]
    norm = Normalize(min(all_values), max(all_values) + 1e-9); cmap = plt.get_cmap('RdYlGn')
    ax.plot(straight_x, straight_y, straight_z, '--', color='white', lw=3.0, label='straight path')
    for radius, azimuth, arc_x, arc_y, arc_z, curved_xc in curved_results:
        ax.plot(arc_x, arc_y, arc_z, color=cmap(norm(curved_xc)), lw=2.2)
    ax.scatter([passer[0], receiver[0]], [passer[1], receiver[1]], [0, 0], c=['#FFDD57', '#57FF8A'], s=80, edgecolors='white')
    for defender in defenders:
        ax.plot3D([defender[0], defender[0]], [defender[1], defender[1]], [0, JUMP_REACH], color=C_DEF, lw=0.8, alpha=0.4, ls='--')
        ax.scatter([defender[0]], [defender[1]], [0], c=C_DEF, s=28, edgecolors='white')
    chord = np.linalg.norm(receiver - passer)
    ax.set_xlim(min(passer[0], receiver[0]) - 10, max(passer[0], receiver[0]) + 10)
    ax.set_ylim(min(passer[1], receiver[1]) - 10, max(passer[1], receiver[1]) + 10)
    ax.set_zlim(0, 8); ax.set_box_aspect((chord, max(chord * 0.8, 12), 8))
    ax.set_xlabel('x (m)', color='#AAA'); ax.set_ylabel('y (m)', color='#AAA'); ax.set_zlabel('z (m)', color='#AAA')
    ax.tick_params(colors='#888'); ax.view_init(elev=25, azim=-58); ax.grid(False)
    scalar = ScalarMappable(norm=norm, cmap=cmap); scalar.set_array(all_values)
    cbar = fig.colorbar(scalar, ax=ax, fraction=0.025, pad=0.02, shrink=0.75)
    cbar.set_label('DAS xC', color='#DDD'); cbar.ax.tick_params(colors='#AAA')

    ax2 = fig.add_axes([0.66, 0.16, 0.30, 0.70]); ax2.set_facecolor('#141820')
    labels = ['original straight', 'curved radius 1.5', 'curved radius 3.0', 'curved radius 4.5', 'curved radius 6.0', 'curved radius 7.5']
    values = [original_xc] + [item[-1] for item in curved_results]
    ax2.barh(np.arange(len(labels)), values, color=['#87939d'] + [cmap(norm(v)) for v in values[1:]], edgecolor='#333')
    for index, value in enumerate(values):
        ax2.text(value + 0.01, index, f'{value:.3f}', va='center', color='#EEE', fontsize=9)
    ax2.set_yticks(np.arange(len(labels))); ax2.set_yticklabels(labels, color='#CCC'); ax2.invert_yaxis()
    ax2.set_xlim(0, max(values) * 1.25); ax2.set_xlabel('DAS expected completion xC', color='#AAA')
    ax2.set_title(f'Original vs genuinely curved DAS\nzero-radius equality error={abs(original_xc - straight_curved_xc):.5f}', color='white', fontsize=11)
    ax2.tick_params(colors='#888'); ax2.grid(axis='x', color='#1a2030', alpha=0.6)
    fig.text(0.5, 0.965, f"Match 3812 event {event_id}: {event['possessionEvents']['passerPlayerName']} -> {event['possessionEvents']['receiverPlayerName']}", ha='center', color='white', fontsize=16, fontweight='bold')
    fig.text(0.5, 0.935, f"tracking frame {frame['frameNum']} | original DAS uses the exact event-frame defensive constellation | straight DAS xC={original_xc:.3f} | curved zero-radius xC={straight_curved_xc:.3f}", ha='center', color='#AAA', fontsize=9.5)
    fig.savefig(output_path, dpi=170, bbox_inches='tight', facecolor=BG)
    plt.close(fig)
    print(f'event={event_id} frame={frame["frameNum"]} original_xC={original_xc:.6f} curved_zero_radius={straight_curved_xc:.6f} error={abs(original_xc - straight_curved_xc):.6f}')

for event_id in TRACKED_EVENT_IDS:
    render_tracked_event_comparison(event_id, f'../figures/das_study/tracked_event_{event_id}_original_vs_curved_das.png')

NameError: name 'ellipsoid_arc_3d' is not defined

## Exact tracked-event 2D DAS heatmap comparison

These are the requested 2D heatmaps using the actual event-start frame from `3812.jsonl`: original DAS `attack_cum_prob` versus the same DAS map weighted point-by-point by the curvature-dependent probability that a curved pass is played.

In [31]:
# ----------------------------------------------------------------------
# Exact tracked-event 2D DAS heatmaps: unconditional original vs behavior-weighted.
# ----------------------------------------------------------------------
import matplotlib.tri as mtri


def exact_event_das_heatmap(event_id, output_path, v0=18.0, n_angles=180):
    event, frame, passer, receiver, defenders = tracked_event_scenario(event_id)
    player_rows = [[passer[0], passer[1], 0.0, 0.0], [receiver[0], receiver[1], 0.0, 0.0]]
    player_rows.extend([[d[0], d[1], 0.0, 0.0] for d in defenders])
    player_pos = np.array([player_rows], dtype=np.float32)
    ball = np.array([[passer[0], passer[1]]], dtype=np.float32)
    phi_values = np.linspace(-np.pi, np.pi, n_angles, endpoint=False, dtype=np.float32)
    das_result = simulate_passes(
        player_pos, ball, phi_values[None, :], np.full((1, n_angles), v0, dtype=np.float32),
        np.array([0]), np.array([0, 0] + [1] * len(defenders)),
        players=np.array(['A', 'B'] + [f'D{i}' for i in range(len(defenders))]),
        passers=np.array(['A']), exclude_passer=True,
        fields_to_return=['attack_cum_prob', 'defense_cum_prob'], radial_gridsize=0.75,
    )
    radii = np.asarray(das_result.r_grid, dtype=float)
    original = np.asarray(das_result.attack_cum_prob[0], dtype=float)
    angle_grid, radius_grid = np.meshgrid(phi_values.astype(float), radii, indexing='ij')
    map_x = passer[0] + radius_grid * np.cos(angle_grid)
    map_y = passer[1] + radius_grid * np.sin(angle_grid)
    weighted = original.copy()
    play_prior_map = np.ones_like(original)
    straight_angle = np.arctan2(receiver[1] - passer[1], receiver[0] - passer[0])
    angle_delta = np.angle(np.exp(1j * (phi_values - straight_angle)))
    straight_angle_index = int(np.argmin(np.abs(angle_delta)))
    for angle_index, phi in enumerate(phi_values):
        for radius_index, distance in enumerate(radii):
            if distance <= 1e-9:
                continue
            endpoint = [passer[0] + distance * np.cos(float(phi)), passer[1] + distance * np.sin(float(phi))]
            curve_x, curve_y, _ = bezier_arc(passer, endpoint, bulge=0.25, n_points=60)
            path_points = np.column_stack([curve_x, curve_y])
            chord_length = np.linalg.norm(path_points[-1] - path_points[0])
            path_length = np.linalg.norm(np.diff(path_points, axis=0), axis=1).sum()
            curvature_score = np.clip((path_length / max(chord_length, 1e-9) - 1.0) / 0.50, 0.0, 1.0)
            play_prior_map[angle_index, radius_index] = curve_play_prior(float(curvature_score))
    # The straight option is not a curved option: use P(straight) = 1 - P(curved).
    straight_play_prior = 1.0 - P_CURVED_BASE
    play_prior_map[straight_angle_index, :] = straight_play_prior
    weighted *= play_prior_map

    straight_radius_index = int(np.argmin(np.abs(radii - np.linalg.norm(receiver - passer))))
    straight_original_xc = float(original[straight_angle_index, straight_radius_index])
    straight_weighted_xc = float(weighted[straight_angle_index, straight_radius_index])
    valid = (map_x >= -52.5) & (map_x <= 52.5) & (map_y >= -34) & (map_y <= 34) & np.isfinite(original)
    flat_valid = valid.ravel()
    flat_x, flat_y = map_x.ravel()[flat_valid], map_y.ravel()[flat_valid]
    triangulation = mtri.Triangulation(flat_x, flat_y)
    original_values = original.ravel()[flat_valid]
    weighted_values = weighted.ravel()[flat_valid]

    event_name = event['possessionEvents']
    fig, axes = plt.subplots(1, 2, figsize=(21, 8), facecolor=BG, constrained_layout=True)
    for panel_index, (axis, values, title, label) in enumerate([
        (axes[0], original_values, 'Original DAS 2D heatmap', 'Original attack_cum_prob'),
        (axes[1], weighted_values, 'Behavior-weighted DAS 2D heatmap', 'DAS xC x P(option played)'),
    ]):
        axis.set_facecolor('#101820')
        mesh = axis.tripcolor(triangulation, values, shading='gouraud', cmap='viridis', vmin=0.0, vmax=1.0)
        draw_pitch_heatmap(axis)
        axis.plot([passer[0], receiver[0]], [passer[1], receiver[1]], color='white', lw=2.8 if panel_index else 1.2,
                  ls='-', alpha=0.95, label=f'Straight xC={straight_original_xc:.3f}' if panel_index else 'straight pass')
        axis.scatter(defenders[:, 0], defenders[:, 1], c=C_DEF, s=40, edgecolors='white', linewidths=0.7)
        axis.scatter([passer[0]], [passer[1]], c='#FFDD57', s=100, edgecolors='white', linewidths=1.2)
        axis.scatter([receiver[0]], [receiver[1]], c='#57FF8A', s=100, edgecolors='white', linewidths=1.2)
        if panel_index:
            axis.text(0.02, 0.97, f'Straight DAS xC={straight_original_xc:.3f}\n'
                      f'P(straight)={straight_play_prior:.3f}\n'
                      f'Weighted straight={straight_weighted_xc:.3f}',
                      transform=axis.transAxes, va='top', color='white', fontsize=9,
                      bbox=dict(facecolor='#141820', edgecolor='#AAA', alpha=0.85))
            axis.legend(loc='upper right', fontsize=8, facecolor='#141820', edgecolor='#333', labelcolor='#DDD')
        axis.set_xlim(-52.5, 52.5); axis.set_ylim(-34, 34); axis.set_aspect('equal')
        axis.set_xlabel('x (m)', color='#AAA'); axis.set_ylabel('y (m)', color='#AAA')
        axis.tick_params(colors='#AAA'); axis.set_title(title, color='white', fontsize=12, fontweight='bold')
        colorbar = fig.colorbar(mesh, ax=axis, fraction=0.046, pad=0.03)
        colorbar.set_label(label, color='#DDD'); colorbar.ax.tick_params(colors='#AAA')
    fig.suptitle(f"Match 3812 event {event_id}: {event_name['passerPlayerName']} -> {event_name['receiverPlayerName']} | frame {frame['frameNum']}", color='white', fontsize=16, fontweight='bold')
    fig.text(0.5, 0.01, 'Original DAS is conditional on an option. The right map weights every option, including straight, by its probability of being played.', ha='center', color='#AAA', fontsize=10)
    fig.savefig(output_path, dpi=170, bbox_inches='tight', facecolor=BG)
    plt.close(fig)
    print(f'Saved {output_path} | straight DAS={straight_original_xc:.5f} | P(straight)={straight_play_prior:.5f} | weighted straight={straight_weighted_xc:.5f}')

for exact_event_id in TRACKED_EVENT_IDS:
    exact_event_das_heatmap(exact_event_id, f'../figures/das_study/tracked_event_{exact_event_id}_2d_das_heatmap_comparison.png')

Saved /mnt/user-data/outputs/tracked_event_6497474_2d_das_heatmap_comparison.png | straight DAS=0.81766 | P(straight)=0.51590 | weighted straight=0.42183
Saved /mnt/user-data/outputs/tracked_event_6497478_2d_das_heatmap_comparison.png | straight DAS=0.48763 | P(straight)=0.51590 | weighted straight=0.25157
Saved /mnt/user-data/outputs/tracked_event_6497488_2d_das_heatmap_comparison.png | straight DAS=0.15291 | P(straight)=0.51590 | weighted straight=0.07888


## Aggregate DAS maps: no starting-position prior versus supplied-grid prior

These maps average the exact tracked event-frame DAS surfaces. The grid-weighted version uses a manually digitized approximation of the supplied `P(straight | starting position)` figure because its underlying numeric table was not provided.

In [11]:
# ----------------------------------------------------------------------
# Aggregate exact tracked-event DAS maps, with and without the supplied
# starting-position straight-pass probability grid.
# ----------------------------------------------------------------------
from scipy.interpolate import griddata

GRID_X = np.linspace(-50, 50, 12)
GRID_Y = np.linspace(-30, 30, 7)
# Approximate percentages read from the supplied 7 x 12 figure.
GRID_P_STRAIGHT = np.array([
    [0.14, 0.18, 0.20, 0.21, 0.25, 0.29, 0.26, 0.23, 0.25, 0.16, 0.15, 0.19],
    [0.10, 0.20, 0.24, 0.24, 0.24, 0.25, 0.27, 0.18, 0.24, 0.20, 0.17, 0.18],
    [0.09, 0.16, 0.18, 0.25, 0.24, 0.24, 0.25, 0.24, 0.24, 0.22, 0.14, 0.16],
    [0.11, 0.18, 0.20, 0.27, 0.27, 0.29, 0.26, 0.25, 0.25, 0.19, 0.15, 0.14],
    [0.16, 0.13, 0.20, 0.27, 0.28, 0.27, 0.25, 0.24, 0.25, 0.22, 0.19, 0.14],
    [0.10, 0.15, 0.25, 0.24, 0.26, 0.28, 0.27, 0.24, 0.24, 0.23, 0.19, 0.17],
    [0.17, 0.22, 0.20, 0.22, 0.21, 0.26, 0.26, 0.22, 0.24, 0.18, 0.18, 0.09],
])


def interpolate_start_prior(x, y):
    return float(np.clip(griddata((GRID_X.repeat(len(GRID_Y)), np.tile(GRID_Y, len(GRID_X))),
                                  GRID_P_STRAIGHT.T.ravel(), (x, y), method='linear'), 0.01, 0.99))


def event_das_points(event_id, n_angles=144):
    event, frame, passer, receiver, defenders = tracked_event_scenario(event_id)
    rows = [[passer[0], passer[1], 0.0, 0.0], [receiver[0], receiver[1], 0.0, 0.0]]
    rows.extend([[d[0], d[1], 0.0, 0.0] for d in defenders])
    result = simulate_passes(
        np.array([rows], dtype=np.float32), np.array([[passer[0], passer[1]]], dtype=np.float32),
        np.linspace(-np.pi, np.pi, n_angles, endpoint=False)[None, :].astype(np.float32),
        np.full((1, n_angles), 18.0, dtype=np.float32), np.array([0]),
        np.array([0, 0] + [1] * len(defenders)), players=np.array(['A', 'B'] + [f'D{i}' for i in range(len(defenders))]),
        passers=np.array(['A']), exclude_passer=True, fields_to_return=['attack_cum_prob'], radial_gridsize=1.0)
    angles = np.linspace(-np.pi, np.pi, n_angles, endpoint=False)
    radii = np.asarray(result.r_grid, dtype=float)
    angle_grid, radius_grid = np.meshgrid(angles, radii, indexing='ij')
    points = np.column_stack([(passer[0] + radius_grid * np.cos(angle_grid)).ravel(),
                              (passer[1] + radius_grid * np.sin(angle_grid)).ravel()])
    values = np.asarray(result.attack_cum_prob[0]).ravel()
    straight_angle = np.arctan2(receiver[1] - passer[1], receiver[0] - passer[0])
    delta = np.angle(np.exp(1j * (angles - straight_angle)))
    straight_index = int(np.argmin(np.abs(delta)))
    local_straight = interpolate_start_prior(passer[0], passer[1])
    option_prior = np.full((n_angles, len(radii)), 1.0 - local_straight)
    option_prior[straight_index, :] = local_straight
    return points, values, option_prior.ravel(), passer, receiver, defenders, frame['frameNum']

common_x = np.linspace(-52.5, 52.5, 105)
common_y = np.linspace(-34, 34, 68)
mesh_x, mesh_y = np.meshgrid(common_x, common_y)
original_maps, weighted_maps, event_labels = [], [], []
for aggregate_event_id in TRACKED_EVENT_IDS:
    points, values, option_prior, passer_pos, receiver_pos, defenders_pos, frame_num = event_das_points(aggregate_event_id)
    original_map = griddata(points, values, (mesh_x, mesh_y), method='linear')
    weighted_map = griddata(points, values * option_prior, (mesh_x, mesh_y), method='linear')
    original_maps.append(original_map); weighted_maps.append(weighted_map)
    event_labels.append(f'{aggregate_event_id} frame {frame_num}')
aggregate_original = np.nanmean(np.stack(original_maps), axis=0)
aggregate_weighted = np.nanmean(np.stack(weighted_maps), axis=0)

fig, axes = plt.subplots(1, 2, figsize=(21, 8), facecolor=BG, constrained_layout=True)
for axis, values, title, label in [
    (axes[0], aggregate_original, 'Aggregate original DAS', 'mean attack_cum_prob'),
    (axes[1], aggregate_weighted, 'Aggregate DAS with starting-position prior', 'mean DAS xC x option-play probability'),
]:
    axis.set_facecolor('#101820')
    image = axis.imshow(values, extent=[-52.5, 52.5, -34, 34], origin='lower', cmap='viridis', vmin=0, vmax=1, aspect='equal')
    draw_pitch_heatmap(axis)
    axis.set_xlabel('x (m)', color='#AAA'); axis.set_ylabel('y (m)', color='#AAA')
    axis.tick_params(colors='#AAA'); axis.set_title(title, color='white', fontsize=13, fontweight='bold')
    colorbar = fig.colorbar(image, ax=axis, fraction=0.046, pad=0.03)
    colorbar.set_label(label, color='#DDD'); colorbar.ax.tick_params(colors='#AAA')
fig.suptitle('Match 3812 aggregate DAS: baseline versus starting-position probability weighted', color='white', fontsize=17, fontweight='bold')
fig.text(0.5, 0.01, 'Events averaged: ' + ', '.join(event_labels) + ' | right map uses an approximate digitization of the supplied starting-position grid.', ha='center', color='#AAA', fontsize=9.5)
fig.savefig('../figures/das_study/match_3812_aggregate_das_without_vs_with_start_prior.png', dpi=170, bbox_inches='tight', facecolor=BG)
plt.close(fig)
print('Saved ../figures/das_study/match_3812_aggregate_das_without_vs_with_start_prior.png')

Saved /mnt/user-data/outputs/match_3812_aggregate_das_without_vs_with_start_prior.png


In [10]:
# Self-contained tracked-event helpers for the aggregate calculation.
import json
TRACKED_EVENT_IDS = [6497474, 6497478, 6497488]
EVENT_PLAYER_SHIRTS = {'Frenkie de Jong': 21, 'Daley Blind': 17, 'Nathan Ake': 4, 'Nathan Aké': 4}
with open('../data/pff_match_data/3812.json', 'r', encoding='utf-8') as event_file:
    _aggregate_metadata = {int(e['gameEventId']): e for e in json.load(event_file)}
_aggregate_frames = {}
with open('../data/pff_match_data/3812.jsonl', 'r', encoding='utf-8') as tracking_file:
    for _line in tracking_file:
        _row = json.loads(_line)
        _event_id = int(_row.get('game_event_id') or -1)
        if _event_id not in TRACKED_EVENT_IDS:
            continue
        _start = int((_row.get('game_event') or {}).get('start_frame') or _row.get('frameNum'))
        if int(_row['frameNum']) == _start:
            _aggregate_frames[_event_id] = _row


def _aggregate_player(players, shirt):
    return next(p for p in players if int(p.get('jerseyNum', -1)) == shirt)


def tracked_event_scenario(event_id):
    event = _aggregate_metadata[event_id]
    possession = event['possessionEvents']
    frame = _aggregate_frames[event_id]
    home_ball = bool(frame['game_event'].get('home_ball'))
    attackers = frame['homePlayers'] if home_ball else frame['awayPlayers']
    defenders = frame['awayPlayers'] if home_ball else frame['homePlayers']
    passer = _aggregate_player(attackers, EVENT_PLAYER_SHIRTS[possession['passerPlayerName']])
    receiver = _aggregate_player(attackers, EVENT_PLAYER_SHIRTS[possession['receiverPlayerName']])
    defender_xy = np.array([[float(p['x']), float(p['y'])] for p in defenders], dtype=float)
    return event, frame, np.array([passer['x'], passer['y']], dtype=float), np.array([receiver['x'], receiver['y']], dtype=float), defender_xy


def draw_pitch_heatmap(axis):
    kw = dict(color='#1f4b2b', lw=1.0, alpha=0.65)
    for x0, x1, y0, y1 in [(-52.5, 52.5, -34, -34), (-52.5, 52.5, 34, 34),
                           (-52.5, -52.5, -34, 34), (52.5, 52.5, -34, 34), (0, 0, -34, 34),
                           (34, 52.5, 16.5, 16.5), (34, 52.5, -16.5, -16.5), (34, 34, -16.5, 16.5)]:
        axis.plot([x0, x1], [y0, y1], **kw)
    theta = np.linspace(0, 2 * np.pi, 100)
    axis.plot(9.15 * np.cos(theta), 9.15 * np.sin(theta), **kw)

## Aggregate original DAS versus true curved-path DAS versus weighted curved DAS

The prior aggregate only weighted the original DAS surface. This section recomputes each non-straight option along an independent 3D curved path, then applies the starting-position play prior. The three panels are: original DAS, conditional curved DAS, and behavior-weighted curved DAS.

In [12]:
# ----------------------------------------------------------------------
# True aggregate curved DAS comparison.
# ----------------------------------------------------------------------
def _aggregate_half_ellipsoid_path(start, end, radius, n_points=50):
    start = np.asarray(start, dtype=float); end = np.asarray(end, dtype=float)
    chord = end - start
    length = np.linalg.norm(chord)
    tangent = chord / max(length, 1e-9)
    lateral = np.array([-tangent[1], tangent[0]])
    theta = np.linspace(np.pi, 0.0, n_points)
    along = 0.5 * (1.0 + np.cos(theta))
    cross = np.sin(theta)
    x = start[0] + chord[0] * along + lateral[0] * radius * cross
    y = start[1] + chord[1] * along + lateral[1] * radius * cross
    z = radius * cross
    return x, y, z


def _true_curved_event_points(event_id, n_angles=36, radius_stride=3):
    event, frame, passer, receiver, defenders = tracked_event_scenario(event_id)
    rows = [[passer[0], passer[1], 0.0, 0.0], [receiver[0], receiver[1], 0.0, 0.0]]
    rows.extend([[d[0], d[1], 0.0, 0.0] for d in defenders])
    angles = np.linspace(-np.pi, np.pi, n_angles, endpoint=False)
    original_result = simulate_passes(
        np.array([rows], dtype=np.float32), np.array([[passer[0], passer[1]]], dtype=np.float32),
        angles[None, :].astype(np.float32), np.full((1, n_angles), 18.0, dtype=np.float32),
        np.array([0]), np.array([0, 0] + [1] * len(defenders)),
        players=np.array(['A', 'B'] + [f'D{i}' for i in range(len(defenders))]),
        passers=np.array(['A']), exclude_passer=True, fields_to_return=['attack_cum_prob'], radial_gridsize=1.25)
    radii = np.asarray(original_result.r_grid, dtype=float)[::radius_stride]
    original_values = np.asarray(original_result.attack_cum_prob[0])[:, ::radius_stride]
    straight_angle = np.arctan2(receiver[1] - passer[1], receiver[0] - passer[0])
    angle_delta = np.angle(np.exp(1j * (angles - straight_angle)))
    straight_index = int(np.argmin(np.abs(angle_delta)))
    local_straight = interpolate_start_prior(passer[0], passer[1])
    curved_values = np.zeros_like(original_values)
    weighted_values = np.zeros_like(original_values)
    for angle_index, phi in enumerate(angles):
        for radius_index, distance in enumerate(radii):
            endpoint = np.array([passer[0] + distance * np.cos(phi), passer[1] + distance * np.sin(phi)])
            if angle_index == straight_index or distance <= 1e-9:
                curved_xc = original_values[angle_index, radius_index]
                option_probability = local_straight
            else:
                arc_x, arc_y, arc_z = _aggregate_half_ellipsoid_path(passer, endpoint, 0.22 * distance)
                curved_xc = compute_xc_curved_3d(
                    passer, receiver, defenders, arc_x, arc_y, arc_z,
                    v0=18.0, jump_reach_height=JUMP_REACH)['xc']
                option_probability = 1.0 - local_straight
            curved_values[angle_index, radius_index] = curved_xc
            weighted_values[angle_index, radius_index] = curved_xc * option_probability
    angle_grid, radius_grid = np.meshgrid(angles, radii, indexing='ij')
    points = np.column_stack([(passer[0] + radius_grid * np.cos(angle_grid)).ravel(),
                              (passer[1] + radius_grid * np.sin(angle_grid)).ravel()])
    return points, original_values.ravel(), curved_values.ravel(), weighted_values.ravel(), frame['frameNum']

original_aggregate_maps = []; curved_aggregate_maps = []; weighted_aggregate_maps = []; aggregate_labels = []
for aggregate_event_id in TRACKED_EVENT_IDS:
    point_set, original_values, curved_values, weighted_values, frame_num = _true_curved_event_points(aggregate_event_id)
    original_aggregate_maps.append(griddata(point_set, original_values, (mesh_x, mesh_y), method='linear'))
    curved_aggregate_maps.append(griddata(point_set, curved_values, (mesh_x, mesh_y), method='linear'))
    weighted_aggregate_maps.append(griddata(point_set, weighted_values, (mesh_x, mesh_y), method='linear'))
    aggregate_labels.append(f'{aggregate_event_id} frame {frame_num}')
mean_original = np.nanmean(np.stack(original_aggregate_maps), axis=0)
mean_curved = np.nanmean(np.stack(curved_aggregate_maps), axis=0)
mean_weighted_curved = np.nanmean(np.stack(weighted_aggregate_maps), axis=0)

fig, axes = plt.subplots(1, 3, figsize=(27, 8), facecolor=BG, constrained_layout=True)
for axis, values, title, label in [
    (axes[0], mean_original, 'Original DAS', 'mean attack_cum_prob'),
    (axes[1], mean_curved, 'Conditional true curved DAS', 'mean curved-path xC'),
    (axes[2], mean_weighted_curved, 'Behavior-weighted curved DAS', 'mean curved xC x option probability'),
]:
    axis.set_facecolor('#101820')
    image = axis.imshow(values, extent=[-52.5, 52.5, -34, 34], origin='lower', cmap='viridis', vmin=0, vmax=1, aspect='equal')
    draw_pitch_heatmap(axis)
    axis.set_xlabel('x (m)', color='#AAA'); axis.set_ylabel('y (m)', color='#AAA')
    axis.tick_params(colors='#AAA'); axis.set_title(title, color='white', fontsize=13, fontweight='bold')
    colorbar = fig.colorbar(image, ax=axis, fraction=0.046, pad=0.03)
    colorbar.set_label(label, color='#DDD'); colorbar.ax.tick_params(colors='#AAA')
fig.suptitle('Match 3812 aggregate DAS: original versus true curved versus weighted curved', color='white', fontsize=17, fontweight='bold')
fig.text(0.5, 0.01, 'Events: ' + ', '.join(aggregate_labels) + ' | 36 directions, true 3D curved-path integration, approximate supplied starting-position grid.', ha='center', color='#AAA', fontsize=9.5)
fig.savefig('../figures/das_study/match_3812_aggregate_original_true_curved_weighted.png', dpi=170, bbox_inches='tight', facecolor=BG)
plt.close(fig)
print('Saved ../figures/das_study/match_3812_aggregate_original_true_curved_weighted.png')

Saved /mnt/user-data/outputs/match_3812_aggregate_original_true_curved_weighted.png


## Single-event three-map comparison

A single-event map describes one defensive constellation at one tracking frame. An aggregate map averages several such surfaces after placing them on a common pitch grid. Therefore the aggregate can be smoother and can contain spatial patterns created by different event locations; it is not the same quantity as any one event.

In [13]:
# ----------------------------------------------------------------------
# Single exact tracked event: original, conditional curved, weighted curved.
# ----------------------------------------------------------------------
SINGLE_EVENT_ID = 6497474
single_event, single_frame, single_passer, single_receiver, single_defenders = tracked_event_scenario(SINGLE_EVENT_ID)
single_points, single_original_values, single_curved_values, single_weighted_values, _ = _true_curved_event_points(SINGLE_EVENT_ID)
single_maps = [
    griddata(single_points, values, (mesh_x, mesh_y), method='linear')
    for values in (single_original_values, single_curved_values, single_weighted_values)
]
single_event_name = single_event['possessionEvents']
fig, axes = plt.subplots(1, 3, figsize=(27, 8), facecolor=BG, constrained_layout=True)
for axis, values, title, label in zip(
    axes, single_maps,
    ['Original DAS', 'Conditional true curved DAS', 'Behavior-weighted curved DAS'],
    ['attack_cum_prob', 'curved-path xC', 'curved xC x option probability']):
    axis.set_facecolor('#101820')
    image = axis.imshow(values, extent=[-52.5, 52.5, -34, 34], origin='lower', cmap='viridis', vmin=0, vmax=1, aspect='equal')
    draw_pitch_heatmap(axis)
    axis.plot([single_passer[0], single_receiver[0]], [single_passer[1], single_receiver[1]], color='white', lw=2.2)
    axis.scatter(single_defenders[:, 0], single_defenders[:, 1], c=C_DEF, s=38, edgecolors='white', linewidths=0.6)
    axis.scatter([single_passer[0]], [single_passer[1]], c='#FFDD57', s=90, edgecolors='white')
    axis.scatter([single_receiver[0]], [single_receiver[1]], c='#57FF8A', s=90, edgecolors='white')
    axis.set_xlabel('x (m)', color='#AAA'); axis.set_ylabel('y (m)', color='#AAA')
    axis.tick_params(colors='#AAA'); axis.set_title(title, color='white', fontsize=13, fontweight='bold')
    colorbar = fig.colorbar(image, ax=axis, fraction=0.046, pad=0.03)
    colorbar.set_label(label, color='#DDD'); colorbar.ax.tick_params(colors='#AAA')
fig.suptitle(f"Match 3812 event {SINGLE_EVENT_ID}: {single_event_name['passerPlayerName']} -> {single_event_name['receiverPlayerName']} | frame {single_frame['frameNum']}", color='white', fontsize=17, fontweight='bold')
fig.text(0.5, 0.01, 'One exact event frame: same players and defenders in all three panels; only path model and option-play weighting change.', ha='center', color='#AAA', fontsize=10)
fig.savefig('../figures/das_study/match_3812_single_event_original_curved_weighted.png', dpi=170, bbox_inches='tight', facecolor=BG)
plt.close(fig)
print('Saved ../figures/das_study/match_3812_single_event_original_curved_weighted.png')

Saved /mnt/user-data/outputs/match_3812_single_event_original_curved_weighted.png


## Aggregate weighting comparison: none, grid, histogram, and combined

The four panels below separate the probability sources:

1. **Unweighted:** original DAS xC only.
2. **Grid-only:** local starting-position probability from the supplied screenshot approximation.
3. **Histogram-only:** global histogram curve rate plus curvature-specific penalty.
4. **Combined:** local grid option probability plus the histogram curvature penalty.

In [14]:
# ----------------------------------------------------------------------
# Aggregate original curved DAS with four probability treatments.
# ----------------------------------------------------------------------
def _weighted_event_maps(event_id, n_angles=36, radius_stride=3):
    event, frame, passer, receiver, defenders = tracked_event_scenario(event_id)
    rows = [[passer[0], passer[1], 0., 0.], [receiver[0], receiver[1], 0., 0.]]
    rows.extend([[d[0], d[1], 0., 0.] for d in defenders])
    angles = np.linspace(-np.pi, np.pi, n_angles, endpoint=False)
    result = simulate_passes(
        np.array([rows], dtype=np.float32), np.array([[passer[0], passer[1]]], dtype=np.float32),
        angles[None, :].astype(np.float32), np.full((1, n_angles), 18., dtype=np.float32),
        np.array([0]), np.array([0, 0] + [1] * len(defenders)),
        players=np.array(['A', 'B'] + [f'D{i}' for i in range(len(defenders))]),
        passers=np.array(['A']), exclude_passer=True, fields_to_return=['attack_cum_prob'], radial_gridsize=1.25)
    radii = np.asarray(result.r_grid, dtype=float)[::radius_stride]
    original = np.asarray(result.attack_cum_prob[0])[:, ::radius_stride]
    straight_angle = np.arctan2(receiver[1] - passer[1], receiver[0] - passer[0])
    angle_delta = np.angle(np.exp(1j * (angles - straight_angle)))
    straight_index = int(np.argmin(np.abs(angle_delta)))
    local_straight = interpolate_start_prior(passer[0], passer[1])
    curved = np.zeros_like(original)
    histogram_weighted = np.zeros_like(original)
    combined = np.zeros_like(original)
    angle_grid, radius_grid = np.meshgrid(angles, radii, indexing='ij')
    for angle_index, phi in enumerate(angles):
        for radius_index, distance in enumerate(radii):
            endpoint = np.array([passer[0] + distance * np.cos(phi), passer[1] + distance * np.sin(phi)])
            if angle_index == straight_index or distance <= 1e-9:
                curved_value = original[angle_index, radius_index]
                histogram_option = 1.0 - P_CURVED_BASE
                combined_option = local_straight
            else:
                arc_x, arc_y, arc_z = _aggregate_half_ellipsoid_path(passer, endpoint, 0.22 * distance)
                curved_value = compute_xc_curved_3d(
                    passer, receiver, defenders, arc_x, arc_y, arc_z,
                    v0=18.0, jump_reach_height=JUMP_REACH)['xc']
                points = np.column_stack([arc_x, arc_y, arc_z])
                chord = np.linalg.norm(points[-1] - points[0])
                path_length = np.linalg.norm(np.diff(points, axis=0), axis=1).sum()
                curvature_score = np.clip((path_length / max(chord, 1e-9) - 1.0) / 0.50, 0.0, 1.0)
                histogram_option = curve_play_prior(float(curvature_score))
                combined_option = (1.0 - local_straight) * np.exp(-1.8 * curvature_score)
            curved[angle_index, radius_index] = curved_value
            histogram_weighted[angle_index, radius_index] = curved_value * histogram_option
            combined[angle_index, radius_index] = curved_value * combined_option
    points = np.column_stack([(passer[0] + radius_grid * np.cos(angle_grid)).ravel(),
                              (passer[1] + radius_grid * np.sin(angle_grid)).ravel()])
    return points, original.ravel(), curved.ravel(), histogram_weighted.ravel(), combined.ravel()

comparison_maps = [[], [], [], []]
for comparison_event_id in TRACKED_EVENT_IDS:
    event_points, unweighted_values, curved_values, histogram_values, combined_values = _weighted_event_maps(comparison_event_id)
    for maps, values in zip(comparison_maps, [unweighted_values, curved_values, histogram_values, combined_values]):
        maps.append(griddata(event_points, values, (mesh_x, mesh_y), method='linear'))
comparison_means = [np.nanmean(np.stack(maps), axis=0) for maps in comparison_maps]

fig, axes = plt.subplots(1, 4, figsize=(35, 8), facecolor=BG, constrained_layout=True)
comparison_titles = ['Unweighted original DAS', 'Conditional true curved DAS', 'Histogram-weighted curved DAS', 'Combined grid + histogram curved DAS']
comparison_labels = ['mean original DAS xC', 'mean curved-path xC', 'mean curved xC x histogram prior', 'mean curved xC x combined prior']
for axis, values, title, label in zip(axes, comparison_means, comparison_titles, comparison_labels):
    axis.set_facecolor('#101820')
    image = axis.imshow(values, extent=[-52.5, 52.5, -34, 34], origin='lower', cmap='viridis', vmin=0, vmax=1, aspect='equal')
    draw_pitch_heatmap(axis)
    axis.set_title(title, color='white', fontsize=12, fontweight='bold')
    axis.set_xlabel('x (m)', color='#AAA'); axis.set_ylabel('y (m)', color='#AAA'); axis.tick_params(colors='#AAA')
    colorbar = fig.colorbar(image, ax=axis, fraction=0.046, pad=0.03)
    colorbar.set_label(label, color='#DDD'); colorbar.ax.tick_params(colors='#AAA')
fig.suptitle('Match 3812 aggregate DAS: probability-source comparison', color='white', fontsize=17, fontweight='bold')
fig.text(0.5, 0.01, 'Histogram prior = 48.41% base curved rate with curvature penalty; grid values are approximate digitization of the supplied image.', ha='center', color='#AAA', fontsize=9.5)
fig.savefig('../figures/das_study/match_3812_aggregate_probability_sources.png', dpi=170, bbox_inches='tight', facecolor=BG)
plt.close(fig)
print('Saved ../figures/das_study/match_3812_aggregate_probability_sources.png')

Saved /mnt/user-data/outputs/match_3812_aggregate_probability_sources.png


## Single-event probability-source comparison

This is the single-event equivalent of the aggregate four-panel figure. It uses event `6497474` and one exact tracking frame, so no averaging across events is involved.

In [15]:
# ----------------------------------------------------------------------
# Single-event four-way probability-source comparison.
# ----------------------------------------------------------------------
single_source_event = 6497474
single_event, single_frame, single_passer, single_receiver, single_defenders = tracked_event_scenario(single_source_event)
single_points, single_unweighted, single_curved, single_histogram, single_combined = _weighted_event_maps(single_source_event)
single_source_values = [single_unweighted, single_curved, single_histogram, single_combined]
single_source_maps = [griddata(single_points, values, (mesh_x, mesh_y), method='linear') for values in single_source_values]
single_name = single_event['possessionEvents']
fig, axes = plt.subplots(1, 4, figsize=(35, 8), facecolor=BG, constrained_layout=True)
source_titles = ['Unweighted original DAS', 'Conditional true curved DAS', 'Histogram-weighted curved DAS', 'Combined grid + histogram curved DAS']
source_labels = ['attack_cum_prob', 'curved-path xC', 'curved xC x histogram prior', 'curved xC x combined prior']
for axis, values, title, label in zip(axes, single_source_maps, source_titles, source_labels):
    axis.set_facecolor('#101820')
    image = axis.imshow(values, extent=[-52.5, 52.5, -34, 34], origin='lower', cmap='viridis', vmin=0, vmax=1, aspect='equal')
    draw_pitch_heatmap(axis)
    axis.plot([single_passer[0], single_receiver[0]], [single_passer[1], single_receiver[1]], color='white', lw=2.2)
    axis.scatter(single_defenders[:, 0], single_defenders[:, 1], c=C_DEF, s=38, edgecolors='white', linewidths=0.6)
    axis.scatter([single_passer[0]], [single_passer[1]], c='#FFDD57', s=90, edgecolors='white')
    axis.scatter([single_receiver[0]], [single_receiver[1]], c='#57FF8A', s=90, edgecolors='white')
    axis.set_title(title, color='white', fontsize=12, fontweight='bold')
    axis.set_xlabel('x (m)', color='#AAA'); axis.set_ylabel('y (m)', color='#AAA'); axis.tick_params(colors='#AAA')
    colorbar = fig.colorbar(image, ax=axis, fraction=0.046, pad=0.03)
    colorbar.set_label(label, color='#DDD'); colorbar.ax.tick_params(colors='#AAA')
fig.suptitle(f"Match 3812 event {single_source_event}: {single_name['passerPlayerName']} -> {single_name['receiverPlayerName']} | frame {single_frame['frameNum']}", color='white', fontsize=17, fontweight='bold')
fig.text(0.5, 0.01, 'One exact event frame; no event averaging. Histogram prior uses 48.41% base curved rate; grid is the approximate supplied starting-position grid.', ha='center', color='#AAA', fontsize=9.5)
fig.savefig('../figures/das_study/match_3812_single_event_probability_sources.png', dpi=170, bbox_inches='tight', facecolor=BG)
plt.close(fig)
print('Saved ../figures/das_study/match_3812_single_event_probability_sources.png')

Saved /mnt/user-data/outputs/match_3812_single_event_probability_sources.png


## Per-pixel DAS histograms: original versus conditional curved

For each exact tracked event, the following histograms use the valid per-pixel values from the original DAS map and the conditional true-curved DAS map. The grid and histogram play priors are intentionally excluded here so this compares the DAS surfaces themselves.

In [16]:
# ----------------------------------------------------------------------
# Per-pixel DAS histograms for all three exact tracked events.
# ----------------------------------------------------------------------
fig, histogram_axes = plt.subplots(1, len(TRACKED_EVENT_IDS), figsize=(21, 6), facecolor=BG, sharey=True)
for histogram_axis, histogram_event_id in zip(histogram_axes, TRACKED_EVENT_IDS):
    event, frame, passer, receiver, defenders = tracked_event_scenario(histogram_event_id)
    points, original_values, curved_values, _, _ = _true_curved_event_points(histogram_event_id)
    valid = np.isfinite(original_values) & np.isfinite(curved_values)
    original_pixel_values = original_values[valid]
    curved_pixel_values = curved_values[valid]
    bins = np.linspace(0.0, max(float(original_pixel_values.max()), float(curved_pixel_values.max())), 26)
    histogram_axis.hist(original_pixel_values, bins=bins, density=True, alpha=0.58, color='#6f8796', label='Original DAS')
    histogram_axis.hist(curved_pixel_values, bins=bins, density=True, alpha=0.58, color='#42b883', label='Conditional curved DAS')
    original_median = np.median(original_pixel_values)
    curved_median = np.median(curved_pixel_values)
    histogram_axis.axvline(original_median, color='#d9e5eb', ls='--', lw=1.3, label=f'Original median={original_median:.3f}')
    histogram_axis.axvline(curved_median, color='#b8ffdb', ls=':', lw=1.5, label=f'Curved median={curved_median:.3f}')
    histogram_axis.set_title(f'Event {histogram_event_id}\nframe {frame["frameNum"]}', color='white', fontsize=11, fontweight='bold')
    histogram_axis.set_xlabel('Per-pixel DAS xC value', color='#AAA')
    histogram_axis.tick_params(colors='#AAA'); histogram_axis.grid(axis='y', color='#30414d', alpha=0.45)
    histogram_axis.set_facecolor('#101820')
    histogram_axis.legend(fontsize=7.5, facecolor='#141820', edgecolor='#333', labelcolor='#DDD')
    print(f'event={histogram_event_id} n={valid.sum()} original median={original_median:.5f} curved median={curved_median:.5f} original p90={np.quantile(original_pixel_values, .90):.5f} curved p90={np.quantile(curved_pixel_values, .90):.5f}')
histogram_axes[0].set_ylabel('Probability density', color='#AAA')
fig.suptitle('Match 3812 per-pixel DAS distributions: original versus conditional curved', color='white', fontsize=16, fontweight='bold')
fig.text(0.5, 0.01, 'Conditional surfaces only: no curve-play grid or histogram prior applied.', ha='center', color='#AAA', fontsize=10)
fig.savefig('../figures/das_study/match_3812_per_pixel_das_histograms_original_vs_curved.png', dpi=170, bbox_inches='tight', facecolor=BG)
plt.close(fig)
print('Saved ../figures/das_study/match_3812_per_pixel_das_histograms_original_vs_curved.png')

event=6497474 n=1260 original median=0.00398 curved median=0.01648 original p90=0.51949 curved p90=0.94832
event=6497478 n=1260 original median=0.00901 curved median=0.01749 original p90=0.30917 curved p90=0.94620
event=6497488 n=1260 original median=0.00270 curved median=0.01514 original p90=0.02579 curved p90=0.83797
Saved /mnt/user-data/outputs/match_3812_per_pixel_das_histograms_original_vs_curved.png


## Pooled per-pixel DAS histograms across 20 Match 3812 events

This pools valid pixel values from at least ten exact tracked events. Each event contributes its original DAS surface and independently recomputed conditional curved DAS surface. No event is duplicated, and no play-probability weighting is applied.

In [ ]:
# ----------------------------------------------------------------------
# Pooled per-pixel histogram across 20 unique mappable Match 3812 events.
# ----------------------------------------------------------------------
with open('../data/pff_match_data/3812.json', 'r', encoding='utf-8') as pooled_event_file:
    pooled_metadata = json.load(pooled_event_file)
pooled_candidates = []
for pooled_event in pooled_metadata:
    pooled_id = int(pooled_event.get('gameEventId', -1))
    pooled_possession = pooled_event.get('possessionEvents') or {}
    passer_name = pooled_possession.get('passerPlayerName')
    receiver_name = pooled_possession.get('receiverPlayerName')
    if (pooled_possession.get('passOutcomeType') == 'C'
            and passer_name in EVENT_PLAYER_SHIRTS
            and receiver_name in EVENT_PLAYER_SHIRTS
            and pooled_id not in pooled_candidates):
        pooled_candidates.append(pooled_id)

pooled_start_frames = {}
with open('../data/pff_match_data/3812.jsonl', 'r', encoding='utf-8') as pooled_tracking_file:
    for pooled_line in pooled_tracking_file:
        pooled_row = json.loads(pooled_line)
        pooled_id = int(pooled_row.get('game_event_id') or -1)
        if pooled_id not in pooled_candidates:
            continue
        pooled_start = int((pooled_row.get('game_event') or {}).get('start_frame') or pooled_row.get('frameNum'))
        if int(pooled_row['frameNum']) == pooled_start:
            pooled_start_frames[pooled_id] = pooled_row
tracked_frames.update(pooled_start_frames)
_aggregate_frames.update(pooled_start_frames)
pooled_ids = [event_id for event_id in pooled_candidates if event_id in _aggregate_frames][:20]

pooled_original_pixels = []
pooled_curved_pixels = []
for pooled_id in pooled_ids:
    _, pooled_original, pooled_curved, _, _ = _true_curved_event_points(pooled_id)
    pooled_original_pixels.append(pooled_original[np.isfinite(pooled_original)])
    pooled_curved_pixels.append(pooled_curved[np.isfinite(pooled_curved)])
pooled_original_pixels = np.concatenate(pooled_original_pixels)
pooled_curved_pixels = np.concatenate(pooled_curved_pixels)
pooled_max = max(float(pooled_original_pixels.max()), float(pooled_curved_pixels.max()))
pooled_bins = np.linspace(0.0, pooled_max, 41)
fig, pooled_axis = plt.subplots(figsize=(12, 7), facecolor=BG)
pooled_axis.set_facecolor('#101820')
pooled_axis.hist(pooled_original_pixels, bins=pooled_bins, density=True, alpha=0.58, color='#6f8796', label='Original DAS')
pooled_axis.hist(pooled_curved_pixels, bins=pooled_bins, density=True, alpha=0.58, color='#42b883', label='Conditional curved DAS')
pooled_axis.axvline(np.median(pooled_original_pixels), color='#d9e5eb', ls='--', lw=1.5, label=f'Original median={np.median(pooled_original_pixels):.3f}')
pooled_axis.axvline(np.median(pooled_curved_pixels), color='#b8ffdb', ls=':', lw=1.8, label=f'Curved median={np.median(pooled_curved_pixels):.3f}')
pooled_axis.set_xlabel('Per-pixel DAS xC value', color='#AAA'); pooled_axis.set_ylabel('Probability density', color='#AAA')
pooled_axis.set_title(f'Match 3812 pooled per-pixel DAS distribution ({len(pooled_ids)} unique events)', color='white', fontsize=15, fontweight='bold')
pooled_axis.tick_params(colors='#AAA'); pooled_axis.grid(axis='y', color='#30414d', alpha=0.45)
pooled_axis.legend(facecolor='#141820', edgecolor='#333', labelcolor='#DDD')
fig.text(0.5, 0.01, f'Original n={len(pooled_original_pixels):,} pixels | Curved n={len(pooled_curved_pixels):,} pixels | events: {", ".join(map(str, pooled_ids))}', ha='center', color='#AAA', fontsize=8)
fig.savefig('../figures/das_study/match_3812_pooled_20_events_per_pixel_histogram_original_vs_curved.png', dpi=170, bbox_inches='tight', facecolor=BG)
plt.close(fig)
print(f'Events used: {len(pooled_ids)}')
print(f'Original pixels: {len(pooled_original_pixels)} | curved pixels: {len(pooled_curved_pixels)}')
print(f'Original median={np.median(pooled_original_pixels):.5f}, p90={np.quantile(pooled_original_pixels, .90):.5f}')
print(f'Curved median={np.median(pooled_curved_pixels):.5f}, p90={np.quantile(pooled_curved_pixels, .90):.5f}')
print('Saved ../figures/das_study/match_3812_pooled_20_events_per_pixel_histogram_original_vs_curved.png')

KeyError: 6497511